In [1]:
#!/usr/bin/env python3
"""
Neuro-Symbolic Social Mind Reasoning Pipeline
Module A1: Person Detection + Tracking (YOLO-face + IoU Tracker)
Module A2: Gaze Detection (GazeLLE)
Module B:  Social Mind Graph Construction
Module C:  Symbolic Theory-of-Mind Reasoning Engine
Module D:  LLM-Grounded Reasoning & Verbalization
"""

import os
import cv2
import torch
import numpy as np
import json
import math
import re
import time
import base64
from io import BytesIO
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple, Any, Set
from collections import defaultdict
from PIL import Image, ImageDraw, ImageFont

from ultralytics import YOLO
import openai

# ================= 环境设置 =================
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ["XFORMERS_DISABLED"] = "1"

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

# ================= API 配置 =================
API_KEY = "sk-bbf9f24ff4194920a43e15749a2dad29"
API_BASE = "https://dashscope.aliyuncs.com/compatible-mode/v1"
MODEL_NAME = "qwen3.6-plus"

client = openai.OpenAI(api_key=API_KEY, base_url=API_BASE)

# ================= 模型加载 =================

# 1. GazeLLE
torch.hub.set_dir(r'/Users/zehao/Desktop/HAI/MultiParty/code/gazelle-main')
gazelle_model, gazelle_transform = torch.hub.load(
    'fkryan/gazelle', 'gazelle_dinov2_vitb14_inout'
)
gazelle_model.eval()
gazelle_model.to(device)

# 2. YOLO Face Detector
face_detector = YOLO('yolov8n-face.pt')


# ================================================================
#                    MODULE A1: Person Detection + Tracking
# ================================================================

@dataclass
class TrackedPerson:
    """一个被跟踪的人物"""
    person_id: int
    bbox: List[float]          # [x1, y1, x2, y2] 像素坐标
    bbox_norm: List[float]     # [x1, y1, x2, y2] 归一化坐标
    confidence: float
    last_seen_frame: int
    # 以下在A2中填充
    gaze_target_pos: Optional[Tuple[float, float]] = None
    gaze_target_person: Optional[int] = None
    gaze_target_type: str = "unknown"    # "person" / "object" / "out_of_frame" / "unknown"
    gaze_heatmap: Optional[np.ndarray] = None
    inout_score: float = 0.0
    head_direction: Optional[Tuple[float, float]] = None
    gaze_confidence: float = 0.0


class PersonTracker:
    """基于IoU的简易多人跟踪器"""

    def __init__(self, iou_threshold=0.3, max_disappear=30, max_persons=10):
        self.iou_threshold = iou_threshold
        self.max_disappear = max_disappear
        self.max_persons = max_persons
        self.next_id = 1
        self.tracked: Dict[int, TrackedPerson] = {}
        self.current_frame = 0

    @staticmethod
    def compute_iou(box1, box2):
        """计算两个bbox的IoU, 输入为 [x1, y1, x2, y2] 像素坐标"""
        x1 = max(box1[0], box2[0])
        y1 = max(box1[1], box2[1])
        x2 = min(box1[2], box2[2])
        y2 = min(box1[3], box2[3])
        inter = max(0, x2 - x1) * max(0, y2 - y1)
        area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
        area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
        union = area1 + area2 - inter
        return inter / union if union > 0 else 0

    def update(self, detections: List[List[float]], confidences: List[float],
               frame_idx: int, img_width: int, img_height: int) -> Dict[int, TrackedPerson]:
        """每帧调用：输入当前帧检测结果，返回更新后的tracked persons"""
        self.current_frame = frame_idx

        active_ids = list(self.tracked.keys())
        active_boxes = [self.tracked[pid].bbox for pid in active_ids]

        matched_det = set()
        matched_track = set()

        if len(active_ids) > 0 and len(detections) > 0:
            iou_matrix = np.zeros((len(active_ids), len(detections)))
            for i, track_box in enumerate(active_boxes):
                for j, det_box in enumerate(detections):
                    iou_matrix[i, j] = self.compute_iou(track_box, det_box)

            while True:
                if iou_matrix.size == 0:
                    break
                max_iou = np.max(iou_matrix)
                if max_iou < self.iou_threshold:
                    break
                i, j = np.unravel_index(np.argmax(iou_matrix), iou_matrix.shape)
                pid = active_ids[i]
                self.tracked[pid].bbox = detections[j]
                self.tracked[pid].bbox_norm = [
                    detections[j][0] / img_width,
                    detections[j][1] / img_height,
                    detections[j][2] / img_width,
                    detections[j][3] / img_height,
                ]
                self.tracked[pid].confidence = confidences[j]
                self.tracked[pid].last_seen_frame = frame_idx
                self._reset_gaze(pid)
                matched_det.add(j)
                matched_track.add(i)
                iou_matrix[i, :] = -1
                iou_matrix[:, j] = -1

        for j in range(len(detections)):
            if j not in matched_det:
                if len(self.tracked) < self.max_persons:
                    pid = self.next_id
                    self.next_id += 1
                    self.tracked[pid] = TrackedPerson(
                        person_id=pid,
                        bbox=detections[j],
                        bbox_norm=[
                            detections[j][0] / img_width,
                            detections[j][1] / img_height,
                            detections[j][2] / img_width,
                            detections[j][3] / img_height,
                        ],
                        confidence=confidences[j],
                        last_seen_frame=frame_idx
                    )

        to_remove = [pid for pid, person in self.tracked.items()
                     if frame_idx - person.last_seen_frame > self.max_disappear]
        for pid in to_remove:
            del self.tracked[pid]

        active = {
            pid: person for pid, person in self.tracked.items()
            if person.last_seen_frame == frame_idx
        }
        return active

    def _reset_gaze(self, pid):
        """重置某人的gaze信息（每帧重新计算）"""
        p = self.tracked[pid]
        p.gaze_target_pos = None
        p.gaze_target_person = None
        p.gaze_target_type = "unknown"
        p.gaze_heatmap = None
        p.inout_score = 0.0
        p.head_direction = None
        p.gaze_confidence = 0.0


# ================================================================
#                    MODULE A2: Gaze Detection with Person IDs
# ================================================================

class GazeDetector:
    """使用GazeLLE检测注视目标，并将注视点映射到Person ID"""

    def __init__(self, model, transform, device,
                 inout_threshold=0.5,
                 gaze_to_person_radius=0.15):
        self.model = model
        self.transform = transform
        self.device = device
        self.inout_threshold = inout_threshold
        self.gaze_to_person_radius = gaze_to_person_radius

    def detect(self, pil_image: Image.Image,
               active_persons: Dict[int, TrackedPerson]) -> Dict[int, TrackedPerson]:
        """对当前帧所有活跃人物进行gaze检测"""
        if len(active_persons) == 0:
            return active_persons

        img_tensor = self.transform(pil_image).unsqueeze(0).to(self.device)

        pid_list = list(active_persons.keys())
        norm_bboxes = [active_persons[pid].bbox_norm for pid in pid_list]
        norm_bboxes_np = [np.array(norm_bboxes)]

        model_input = {"images": img_tensor, "bboxes": norm_bboxes_np}

        with torch.no_grad():
            output = self.model(model_input)

        heatmaps = output['heatmap'][0]
        inout_scores = output['inout'][0] if output['inout'] is not None else None

        for i, pid in enumerate(pid_list):
            person = active_persons[pid]
            heatmap = heatmaps[i].detach().cpu().numpy()
            inout = inout_scores[i].item() if inout_scores is not None else 1.0

            person.inout_score = inout
            person.gaze_heatmap = heatmap

            if inout > self.inout_threshold:
                max_idx = np.unravel_index(np.argmax(heatmap), heatmap.shape)
                gaze_y = max_idx[0] / heatmap.shape[0]
                gaze_x = max_idx[1] / heatmap.shape[1]

                person.gaze_target_pos = (gaze_x, gaze_y)
                person.gaze_confidence = float(np.max(heatmap))

                face_cx = (person.bbox_norm[0] + person.bbox_norm[2]) / 2
                face_cy = (person.bbox_norm[1] + person.bbox_norm[3]) / 2
                dx = gaze_x - face_cx
                dy = gaze_y - face_cy
                norm = math.sqrt(dx ** 2 + dy ** 2) + 1e-8
                person.head_direction = (dx / norm, dy / norm)

                person.gaze_target_person, person.gaze_target_type = \
                    self._map_gaze_to_person(gaze_x, gaze_y, pid, active_persons)
            else:
                person.gaze_target_type = "out_of_frame"
                person.gaze_confidence = inout

        return active_persons

    def _map_gaze_to_person(self, gaze_x: float, gaze_y: float,
                            self_pid: int,
                            active_persons: Dict[int, TrackedPerson]
                            ) -> Tuple[Optional[int], str]:
        """将注视点映射到最近的人物"""
        best_pid = None
        best_dist = float('inf')

        for pid, person in active_persons.items():
            if pid == self_pid:
                continue
            bx1, by1, bx2, by2 = person.bbox_norm
            expand = 0.05
            bx1_e, by1_e = max(0, bx1 - expand), max(0, by1 - expand)
            bx2_e, by2_e = min(1, bx2 + expand), min(1, by2 + expand)

            cx = (bx1 + bx2) / 2
            cy = (by1 + by2) / 2
            dist = math.sqrt((gaze_x - cx) ** 2 + (gaze_y - cy) ** 2)

            if bx1_e <= gaze_x <= bx2_e and by1_e <= gaze_y <= by2_e:
                if dist < best_dist:
                    best_dist = dist
                    best_pid = pid
            else:
                if dist < self.gaze_to_person_radius and dist < best_dist:
                    best_dist = dist
                    best_pid = pid

        if best_pid is not None:
            return best_pid, "person"
        else:
            return None, "object"


# ================================================================
#               MODULE B: Social Mind Graph Construction
# ================================================================

@dataclass
class GraphNode:
    """Social Mind Graph的节点"""
    person_id: int
    observable_state: dict
    confidence: dict

@dataclass
class GraphEdge:
    """Social Mind Graph的边（可观察关系）"""
    source_id: int
    target_id: int
    edge_type: str          # "GAZES_AT", "PROXIMATE"
    confidence: float
    attributes: dict = field(default_factory=dict)

@dataclass
class DerivedRelation:
    """Social Mind Graph的派生关系"""
    relation_type: str
    participants: List[int]
    value: bool
    confidence: float
    reasoning: str


class SocialMindGraph:
    """Social Mind Graph：形式化的社交场景表示"""

    def __init__(self, fov_angle_deg=150):
        self.fov_angle = math.radians(fov_angle_deg)
        self.nodes: Dict[int, GraphNode] = {}
        self.edges: List[GraphEdge] = []
        self.derived_relations: List[DerivedRelation] = []
        self.visual_access_matrix: Dict[Tuple[int, int], dict] = {}
        self.gazes_at: Dict[int, Optional[int]] = {}
        self.gazed_by: Dict[int, List[int]] = defaultdict(list)

    def build(self, active_persons: Dict[int, TrackedPerson],
              img_width: int, img_height: int):
        """从Module A输出构建完整的Social Mind Graph"""
        self._clear()

        if len(active_persons) < 2:
            for pid, person in active_persons.items():
                self._build_node(person)
            return

        for pid, person in active_persons.items():
            self._build_node(person)

        self._build_observable_edges(active_persons, img_width, img_height)
        self._compute_visual_access(active_persons)
        self._derive_relations(active_persons)

    def _clear(self):
        self.nodes.clear()
        self.edges.clear()
        self.derived_relations.clear()
        self.visual_access_matrix.clear()
        self.gazes_at.clear()
        self.gazed_by.clear()

    # ---- Step 1: Node Construction ----

    def _build_node(self, person: TrackedPerson):
        node = GraphNode(
            person_id=person.person_id,
            observable_state={
                "position": {
                    "bbox_norm": person.bbox_norm,
                    "center": (
                        (person.bbox_norm[0] + person.bbox_norm[2]) / 2,
                        (person.bbox_norm[1] + person.bbox_norm[3]) / 2
                    )
                },
                "gaze": {
                    "target_person": person.gaze_target_person,
                    "target_position": person.gaze_target_pos,
                    "target_type": person.gaze_target_type,
                    "inout_score": person.inout_score
                },
                "head_direction": person.head_direction,
            },
            confidence={
                "detection": person.confidence,
                "gaze": person.gaze_confidence
            }
        )
        self.nodes[person.person_id] = node

    # ---- Step 2: Observable Edges ----

    def _build_observable_edges(self, active_persons: Dict[int, TrackedPerson],
                                img_width: int, img_height: int):
        pids = list(active_persons.keys())

        for pid in pids:
            person = active_persons[pid]
            if person.gaze_target_person is not None:
                target_pid = person.gaze_target_person
                edge = GraphEdge(
                    source_id=pid, target_id=target_pid,
                    edge_type="GAZES_AT",
                    confidence=person.gaze_confidence,
                    attributes={
                        "gaze_point": person.gaze_target_pos,
                        "inout_score": person.inout_score
                    }
                )
                self.edges.append(edge)
                self.gazes_at[pid] = target_pid
                self.gazed_by[target_pid].append(pid)
            else:
                self.gazes_at[pid] = None

        for i in range(len(pids)):
            for j in range(i + 1, len(pids)):
                pi = active_persons[pids[i]]
                pj = active_persons[pids[j]]
                ci = ((pi.bbox_norm[0] + pi.bbox_norm[2]) / 2,
                      (pi.bbox_norm[1] + pi.bbox_norm[3]) / 2)
                cj = ((pj.bbox_norm[0] + pj.bbox_norm[2]) / 2,
                      (pj.bbox_norm[1] + pj.bbox_norm[3]) / 2)
                dist = math.sqrt((ci[0] - cj[0]) ** 2 + (ci[1] - cj[1]) ** 2)

                if dist < 0.15:
                    level = "intimate"
                elif dist < 0.30:
                    level = "personal"
                elif dist < 0.55:
                    level = "social"
                else:
                    level = "public"

                self.edges.append(GraphEdge(
                    source_id=pids[i], target_id=pids[j],
                    edge_type="PROXIMATE",
                    confidence=min(pi.confidence, pj.confidence),
                    attributes={"distance_norm": dist, "level": level}
                ))

    # ---- Step 3: Visual Access Matrix ----

    def _compute_visual_access(self, active_persons: Dict[int, TrackedPerson]):
        pids = list(active_persons.keys())
        for pid_i in pids:
            pi = active_persons[pid_i]
            ci = ((pi.bbox_norm[0] + pi.bbox_norm[2]) / 2,
                  (pi.bbox_norm[1] + pi.bbox_norm[3]) / 2)
            for pid_j in pids:
                if pid_i == pid_j:
                    continue
                pj = active_persons[pid_j]
                cj = ((pj.bbox_norm[0] + pj.bbox_norm[2]) / 2,
                      (pj.bbox_norm[1] + pj.bbox_norm[3]) / 2)
                can_observe, conf, reason = self._check_visual_access(pi, ci, pj, cj)
                self.visual_access_matrix[(pid_i, pid_j)] = {
                    "can_observe": can_observe,
                    "confidence": conf,
                    "reasoning": reason
                }

    def _check_visual_access(self, person_i: TrackedPerson,
                             center_i: Tuple[float, float],
                             person_j: TrackedPerson,
                             center_j: Tuple[float, float]
                             ) -> Tuple[bool, float, str]:
        if person_i.head_direction is None:
            return True, 0.3, "Head direction unknown, assuming possible visibility"

        hd_x, hd_y = person_i.head_direction
        dir_x = center_j[0] - center_i[0]
        dir_y = center_j[1] - center_i[1]
        dir_norm = math.sqrt(dir_x ** 2 + dir_y ** 2) + 1e-8
        dir_x /= dir_norm
        dir_y /= dir_norm

        cos_angle = max(-1, min(1, hd_x * dir_x + hd_y * dir_y))
        angle = math.acos(cos_angle)
        half_fov = self.fov_angle / 2

        if angle <= half_fov * 0.5:
            return True, 0.9, f"P{person_j.person_id} is in central FOV of P{person_i.person_id} (angle={math.degrees(angle):.0f}°)"
        elif angle <= half_fov:
            return True, 0.6, f"P{person_j.person_id} is in peripheral FOV of P{person_i.person_id} (angle={math.degrees(angle):.0f}°)"
        else:
            return False, 0.8, f"P{person_j.person_id} is outside FOV of P{person_i.person_id} (angle={math.degrees(angle):.0f}°)"

    # ---- Step 4: Derived Relations ----

    def _derive_relations(self, active_persons: Dict[int, TrackedPerson]):
        pids = list(active_persons.keys())

        for i in range(len(pids)):
            for j in range(len(pids)):
                if i == j:
                    continue
                pid_i, pid_j = pids[i], pids[j]
                va = self.visual_access_matrix.get((pid_i, pid_j))
                if va:
                    self.derived_relations.append(DerivedRelation(
                        relation_type="CAN_OBSERVE",
                        participants=[pid_i, pid_j],
                        value=va["can_observe"],
                        confidence=va["confidence"],
                        reasoning=va["reasoning"]
                    ))

        for i in range(len(pids)):
            for j in range(i + 1, len(pids)):
                pid_i, pid_j = pids[i], pids[j]
                if self.gazes_at.get(pid_i) == pid_j and self.gazes_at.get(pid_j) == pid_i:
                    conf = min(active_persons[pid_i].gaze_confidence,
                               active_persons[pid_j].gaze_confidence)
                    self.derived_relations.append(DerivedRelation(
                        relation_type="MUTUAL_GAZE",
                        participants=[pid_i, pid_j], value=True,
                        confidence=conf,
                        reasoning=f"P{pid_i} gazes at P{pid_j} AND P{pid_j} gazes at P{pid_i}"
                    ))

        for pid_i in pids:
            target = self.gazes_at.get(pid_i)
            if target is not None and self.gazes_at.get(target) != pid_i:
                self.derived_relations.append(DerivedRelation(
                    relation_type="UNRECIPROCATED_ENGAGEMENT",
                    participants=[pid_i, target], value=True,
                    confidence=active_persons[pid_i].gaze_confidence,
                    reasoning=f"P{pid_i} gazes at P{target}, but P{target} does not gaze back at P{pid_i}"
                ))

        for pid in pids:
            watchers = self.gazed_by.get(pid, [])
            if len(watchers) >= 2:
                self.derived_relations.append(DerivedRelation(
                    relation_type="ATTENTION_CENTER",
                    participants=[pid] + watchers, value=True,
                    confidence=0.7,
                    reasoning=f"P{pid} is gazed at by {len(watchers)} people: {['P' + str(w) for w in watchers]}"
                ))

        for pid in pids:
            watchers = self.gazed_by.get(pid, [])
            target = self.gazes_at.get(pid)
            if len(watchers) == 0 and (target is None or self.gazes_at.get(target) != pid):
                if len(pids) > 2:
                    self.derived_relations.append(DerivedRelation(
                        relation_type="POSSIBLY_EXCLUDED",
                        participants=[pid], value=True, confidence=0.5,
                        reasoning=f"P{pid} is not gazed at by anyone and has no reciprocated engagement"
                    ))

    # ---- 输出与可视化 ----

    def to_dict(self) -> dict:
        return {
            "nodes": {
                pid: {
                    "person_id": node.person_id,
                    "observable_state": node.observable_state,
                    "confidence": node.confidence
                }
                for pid, node in self.nodes.items()
            },
            "observable_edges": [
                {
                    "source": e.source_id, "target": e.target_id,
                    "type": e.edge_type, "confidence": e.confidence,
                    "attributes": e.attributes
                }
                for e in self.edges
            ],
            "visual_access_matrix": {
                f"P{k[0]}->P{k[1]}": v
                for k, v in self.visual_access_matrix.items()
            },
            "derived_relations": [
                {
                    "type": r.relation_type,
                    "participants": [f"P{p}" for p in r.participants],
                    "value": r.value, "confidence": r.confidence,
                    "reasoning": r.reasoning
                }
                for r in self.derived_relations
            ],
            "summary": self._generate_summary()
        }

    def _generate_summary(self) -> dict:
        gaze_pairs = [f"P{pid}→P{t}" for pid, t in self.gazes_at.items() if t is not None]
        mutual = [r for r in self.derived_relations if r.relation_type == "MUTUAL_GAZE"]
        unrecip = [r for r in self.derived_relations if r.relation_type == "UNRECIPROCATED_ENGAGEMENT"]
        return {
            "num_persons": len(self.nodes),
            "gaze_interactions": gaze_pairs,
            "mutual_gaze_pairs": [f"P{r.participants[0]}↔P{r.participants[1]}" for r in mutual],
            "unreciprocated": [f"P{r.participants[0]}→P{r.participants[1]}(unrecip)" for r in unrecip],
        }

    def to_natural_language(self) -> str:
        if len(self.nodes) == 0:
            return "No persons detected in the scene."

        lines = [f"=== Social Mind Graph ({len(self.nodes)} persons) ===\n"]

        lines.append("[Persons & Positions]")
        for pid, node in sorted(self.nodes.items()):
            pos = node.observable_state["position"]["center"]
            lines.append(f"  Person P{pid}: center=({pos[0]:.2f}, {pos[1]:.2f})")

        lines.append("\n[Gaze Relationships]")
        for pid in sorted(self.gazes_at.keys()):
            target = self.gazes_at[pid]
            person = self.nodes[pid]
            if target is not None:
                conf = person.confidence.get("gaze", 0)
                lines.append(f"  P{pid} GAZES_AT P{target} (conf={conf:.2f})")
            else:
                gaze_info = person.observable_state["gaze"]
                lines.append(f"  P{pid} gazes at {gaze_info['target_type']} (not a person)")

        lines.append("\n[Visual Access Matrix]")
        pids = sorted(self.nodes.keys())
        for pid_i in pids:
            can_see, cannot_see = [], []
            for pid_j in pids:
                if pid_i == pid_j:
                    continue
                va = self.visual_access_matrix.get((pid_i, pid_j))
                if va and va["can_observe"]:
                    can_see.append(f"P{pid_j}")
                elif va and not va["can_observe"]:
                    cannot_see.append(f"P{pid_j}")
            if can_see:
                lines.append(f"  P{pid_i} CAN observe: {', '.join(can_see)}")
            if cannot_see:
                lines.append(f"  P{pid_i} CANNOT observe: {', '.join(cannot_see)}")

        lines.append("\n[Derived Social Relations]")
        for r in self.derived_relations:
            participants_str = ", ".join([f"P{p}" for p in r.participants])
            status = "✓" if r.value else "✗"
            lines.append(f"  {status} {r.relation_type}({participants_str}) [conf={r.confidence:.2f}]")
            lines.append(f"    Reasoning: {r.reasoning}")

        return "\n".join(lines)


# ================================================================
#               可视化函数
# ================================================================

def visualize_with_ids(pil_image: Image.Image,
                       active_persons: Dict[int, TrackedPerson],
                       graph: SocialMindGraph,
                       inout_thresh: float = 0.5) -> Image.Image:
    """增强版可视化：显示Person ID + 注视线 + 关系标注"""
    colors_map = {
        1: 'lime', 2: 'tomato', 3: 'cyan',
        4: 'fuchsia', 5: 'yellow', 6: 'orange',
        7: 'white', 8: 'deepskyblue', 9: 'chartreuse', 10: 'hotpink'
    }

    overlay = pil_image.copy().convert("RGBA")
    draw = ImageDraw.Draw(overlay)
    width, height = pil_image.size

    try:
        font_large = ImageFont.truetype("arial.ttf", int(min(width, height) * 0.035))
        font_small = ImageFont.truetype("arial.ttf", int(min(width, height) * 0.02))
    except Exception:
        font_large = ImageFont.load_default()
        font_small = ImageFont.load_default()

    for pid, person in active_persons.items():
        color = colors_map.get(pid % 10 + 1, 'white')
        x1, y1, x2, y2 = person.bbox_norm
        px1, py1 = x1 * width, y1 * height
        px2, py2 = x2 * width, y2 * height

        draw.rectangle([px1, py1, px2, py2], outline=color, width=3)
        draw.text((px1, max(0, py1 - 25)), f"P{pid}", fill=color, font=font_large)

        if person.inout_score > inout_thresh and person.gaze_target_pos:
            gx = person.gaze_target_pos[0] * width
            gy = person.gaze_target_pos[1] * height
            cx = (px1 + px2) / 2
            cy = (py1 + py2) / 2
            draw.line([(cx, cy), (gx, gy)], fill=color, width=2)
            draw.ellipse([(gx - 6, gy - 6), (gx + 6, gy + 6)], fill=color)
            if person.gaze_target_person is not None:
                target_label = f"→P{person.gaze_target_person}"
            else:
                target_label = f"→{person.gaze_target_type}"
            draw.text((px2 + 3, py1), target_label, fill=color, font=font_small)

    for rel in graph.derived_relations:
        if rel.relation_type == "MUTUAL_GAZE" and rel.value:
            p1, p2 = rel.participants
            if p1 in active_persons and p2 in active_persons:
                c1 = ((active_persons[p1].bbox_norm[0] + active_persons[p1].bbox_norm[2]) / 2 * width,
                      (active_persons[p1].bbox_norm[1] + active_persons[p1].bbox_norm[3]) / 2 * height)
                c2 = ((active_persons[p2].bbox_norm[0] + active_persons[p2].bbox_norm[2]) / 2 * width,
                      (active_persons[p2].bbox_norm[1] + active_persons[p2].bbox_norm[3]) / 2 * height)
                draw.line([c1, c2], fill='gold', width=4)
                mid = ((c1[0] + c2[0]) / 2, (c1[1] + c2[1]) / 2)
                draw.text(mid, "MUTUAL", fill='gold', font=font_small)

    return overlay.convert("RGB")


# ================================================================
#               MODULE C: Symbolic ToM Reasoning Engine
# ================================================================

@dataclass
class MentalStateEntry:
    """心智状态知识库中的一条条目"""
    entry_id: str
    order: int
    entry_type: str              # OBSERVABLE / BELIEF / IGNORANCE / FEELING / INTENT /
                                 # RECURSIVE_BELIEF / SOCIAL_DYNAMIC
    subject: int
    predicate: str
    about: Optional[int] = None
    value: bool = True
    confidence: float = 1.0
    rule_applied: str = ""
    premises: List[str] = field(default_factory=list)
    natural_language: str = ""


class MentalStateKnowledgeBase:
    """心智状态知识库 (MSKB)"""

    def __init__(self):
        self.entries: Dict[str, MentalStateEntry] = {}
        self.by_order: Dict[int, List[str]] = defaultdict(list)
        self.by_subject: Dict[int, List[str]] = defaultdict(list)
        self._counter = 0

    def add(self, entry: MentalStateEntry) -> str:
        self._counter += 1
        entry.entry_id = f"MS_{entry.order}_{self._counter}"
        self.entries[entry.entry_id] = entry
        self.by_order[entry.order].append(entry.entry_id)
        self.by_subject[entry.subject].append(entry.entry_id)
        return entry.entry_id

    def get_by_order(self, order: int) -> List[MentalStateEntry]:
        return [self.entries[eid] for eid in self.by_order.get(order, [])]

    def get_by_subject(self, subject: int) -> List[MentalStateEntry]:
        return [self.entries[eid] for eid in self.by_subject.get(subject, [])]

    def to_dict(self) -> dict:
        result = {}
        for order in sorted(self.by_order.keys()):
            result[f"order_{order}"] = []
            for eid in self.by_order[order]:
                e = self.entries[eid]
                result[f"order_{order}"].append({
                    "id": e.entry_id, "type": e.entry_type,
                    "subject": f"P{e.subject}",
                    "about": f"P{e.about}" if e.about else None,
                    "predicate": e.predicate, "value": e.value,
                    "confidence": round(e.confidence, 3),
                    "rule": e.rule_applied, "premises": e.premises,
                    "natural_language": e.natural_language
                })
        return result

    def to_natural_language(self, min_confidence: float = 0.3) -> str:
        lines = ["=" * 60, "  MENTAL STATE KNOWLEDGE BASE", "=" * 60]

        order_names = {
            0: "Observable States (what can be seen)",
            1: "1st-Order Beliefs (what each person knows/feels/wants)",
            2: "2nd-Order Beliefs (what each person thinks others know)",
            3: "3rd-Order Beliefs (recursive meta-beliefs)"
        }

        for order in sorted(self.by_order.keys()):
            header = order_names.get(order, f"Order-{order} Mental States")
            lines.append(f"\n--- {header} ---")
            entries = [self.entries[eid] for eid in self.by_order[order]
                       if self.entries[eid].confidence >= min_confidence]
            if not entries:
                lines.append("  (none)")
                continue
            for e in entries:
                conf_label = ("certain" if e.confidence > 0.8 else
                              "likely" if e.confidence > 0.6 else
                              "possible" if e.confidence > 0.4 else "uncertain")
                marker = "✓" if e.value else "✗"
                lines.append(f"  {marker} [{conf_label}] {e.natural_language}")
                if e.rule_applied:
                    lines.append(f"      Rule: {e.rule_applied}")

        dynamics = [self.entries[eid] for eid in self.entries
                    if self.entries[eid].entry_type == "SOCIAL_DYNAMIC"]
        if dynamics:
            lines.append("\n--- Social Dynamics ---")
            for d in dynamics:
                lines.append(f"  ★ {d.natural_language}")

        return "\n".join(lines)


class SymbolicToMEngine:
    """符号ToM推理引擎"""

    def __init__(self, max_order: int = 3, confidence_decay: float = 0.85):
        self.max_order = max_order
        self.confidence_decay = confidence_decay
        self.mskb = MentalStateKnowledgeBase()
        self.graph = None

    def reason(self, graph: SocialMindGraph) -> MentalStateKnowledgeBase:
        """主推理函数"""
        self.mskb = MentalStateKnowledgeBase()
        self.graph = graph

        if len(graph.nodes) < 1:
            return self.mskb

        self._reason_order_0()

        if len(graph.nodes) < 2:
            return self.mskb

        self._reason_order_1()
        self._reason_order_2()

        for order in range(3, self.max_order + 1):
            if self._reason_higher_order(order) == 0:
                break

        self._infer_social_dynamics()
        return self.mskb

    # ---- 0阶推理 ----

    def _reason_order_0(self):
        for pid, node in self.graph.nodes.items():
            gaze_info = node.observable_state.get("gaze", {})
            target_person = gaze_info.get("target_person")
            target_type = gaze_info.get("target_type", "unknown")

            if target_person is not None:
                self.mskb.add(MentalStateEntry(
                    entry_id="", order=0, entry_type="OBSERVABLE",
                    subject=pid, predicate=f"gazes_at_P{target_person}",
                    about=target_person, value=True,
                    confidence=node.confidence.get("gaze", 0.5),
                    rule_applied="Direct Observation",
                    natural_language=f"P{pid} is looking at P{target_person}"
                ))
            elif target_type == "object":
                self.mskb.add(MentalStateEntry(
                    entry_id="", order=0, entry_type="OBSERVABLE",
                    subject=pid, predicate="gazes_at_object",
                    value=True,
                    confidence=node.confidence.get("gaze", 0.5),
                    rule_applied="Direct Observation",
                    natural_language=f"P{pid} is looking at an object/location (not at any person)"
                ))
            elif target_type == "out_of_frame":
                self.mskb.add(MentalStateEntry(
                    entry_id="", order=0, entry_type="OBSERVABLE",
                    subject=pid, predicate="gazes_out_of_frame",
                    value=True,
                    confidence=node.confidence.get("gaze", 0.5),
                    rule_applied="Direct Observation",
                    natural_language=f"P{pid} is looking outside the frame"
                ))

            watchers = self.graph.gazed_by.get(pid, [])
            if watchers:
                self.mskb.add(MentalStateEntry(
                    entry_id="", order=0, entry_type="OBSERVABLE",
                    subject=pid, predicate=f"gazed_by_{len(watchers)}_people",
                    value=True, confidence=0.8,
                    rule_applied="Aggregation",
                    natural_language=f"P{pid} is being looked at by {len(watchers)} person(s): {['P' + str(w) for w in watchers]}"
                ))

    # ---- 1阶推理 ----

    def _reason_order_1(self):
        pids = list(self.graph.nodes.keys())

        for pid_i in pids:
            for pid_j in pids:
                if pid_i == pid_j:
                    continue
                va = self.graph.visual_access_matrix.get((pid_i, pid_j))
                if va is None:
                    continue

                can_observe = va["can_observe"]
                va_conf = va["confidence"]
                j_gaze_target = self.graph.gazes_at.get(pid_j)
                j_node = self.graph.nodes.get(pid_j)

                # Rule 1.1
                if can_observe and j_node:
                    j_gaze_type = j_node.observable_state.get("gaze", {}).get("target_type", "unknown")
                    if j_gaze_target is not None:
                        conf = va_conf * j_node.confidence.get("gaze", 0.5)
                        self.mskb.add(MentalStateEntry(
                            entry_id="", order=1, entry_type="BELIEF",
                            subject=pid_i,
                            predicate=f"aware_that_P{pid_j}_looks_at_P{j_gaze_target}",
                            about=pid_j, value=True, confidence=conf,
                            rule_applied="Rule 1.1: Visual Access → Awareness",
                            natural_language=f"P{pid_i} can see that P{pid_j} is looking at P{j_gaze_target}"
                        ))
                    elif j_gaze_type == "object":
                        conf = va_conf * 0.7
                        self.mskb.add(MentalStateEntry(
                            entry_id="", order=1, entry_type="BELIEF",
                            subject=pid_i,
                            predicate=f"aware_that_P{pid_j}_looks_at_object",
                            about=pid_j, value=True, confidence=conf,
                            rule_applied="Rule 1.1: Visual Access → Awareness",
                            natural_language=f"P{pid_i} can see that P{pid_j} is looking at something (not a person)"
                        ))

                # Rule 1.2
                if not can_observe:
                    if j_gaze_target is not None:
                        self.mskb.add(MentalStateEntry(
                            entry_id="", order=1, entry_type="IGNORANCE",
                            subject=pid_i,
                            predicate=f"unaware_that_P{pid_j}_looks_at_P{j_gaze_target}",
                            about=pid_j, value=True, confidence=va_conf,
                            rule_applied="Rule 1.2: No Visual Access → Ignorance",
                            natural_language=f"P{pid_i} does NOT know that P{pid_j} is looking at P{j_gaze_target} (P{pid_j} is outside P{pid_i}'s field of view)"
                        ))
                    if j_gaze_target == pid_i:
                        conf = va_conf * self.graph.nodes[pid_j].confidence.get("gaze", 0.5)
                        self.mskb.add(MentalStateEntry(
                            entry_id="", order=1, entry_type="IGNORANCE",
                            subject=pid_i,
                            predicate=f"unaware_of_being_watched_by_P{pid_j}",
                            about=pid_j, value=True, confidence=conf,
                            rule_applied="Rule 1.2b: No Access + Being Watched → Unaware of Surveillance",
                            natural_language=f"P{pid_i} does NOT know that P{pid_j} is watching them (P{pid_i} cannot see P{pid_j})"
                        ))

        # Rule 1.3
        for rel in self.graph.derived_relations:
            if rel.relation_type == "UNRECIPROCATED_ENGAGEMENT" and rel.value:
                pid_i, pid_j = rel.participants
                va = self.graph.visual_access_matrix.get((pid_i, pid_j))
                if va and va["can_observe"]:
                    self.mskb.add(MentalStateEntry(
                        entry_id="", order=1, entry_type="FEELING",
                        subject=pid_i,
                        predicate=f"may_feel_ignored_or_waiting_for_P{pid_j}",
                        about=pid_j, value=True, confidence=rel.confidence * 0.7,
                        rule_applied="Rule 1.3: Unreciprocated Engagement → Possible Feeling",
                        natural_language=f"P{pid_i} is engaging P{pid_j} but P{pid_j} is not looking back — P{pid_i} may feel ignored or be waiting for P{pid_j}'s attention"
                    ))

        # Rule 1.4
        for rel in self.graph.derived_relations:
            if rel.relation_type == "MUTUAL_GAZE" and rel.value:
                pid_i, pid_j = rel.participants
                for a, b in [(pid_i, pid_j), (pid_j, pid_i)]:
                    self.mskb.add(MentalStateEntry(
                        entry_id="", order=1, entry_type="BELIEF",
                        subject=a, predicate=f"mutually_engaged_with_P{b}",
                        about=b, value=True, confidence=rel.confidence,
                        rule_applied="Rule 1.4: Mutual Gaze → Shared Engagement",
                        natural_language=f"P{a} and P{b} are mutually engaged (both looking at each other)"
                    ))

        # Rule 1.5
        for pid_i in pids:
            for watcher_pid in self.graph.gazed_by.get(pid_i, []):
                va = self.graph.visual_access_matrix.get((pid_i, watcher_pid))
                if va and va["can_observe"]:
                    self.mskb.add(MentalStateEntry(
                        entry_id="", order=1, entry_type="BELIEF",
                        subject=pid_i,
                        predicate=f"aware_of_being_watched_by_P{watcher_pid}",
                        about=watcher_pid, value=True,
                        confidence=va["confidence"] * 0.8,
                        rule_applied="Rule 1.5: Being Gazed At + Visual Access → Aware of Being Watched",
                        natural_language=f"P{pid_i} can see that P{watcher_pid} is looking at them, so P{pid_i} knows P{watcher_pid} is watching"
                    ))

        # Rule 1.6
        for rel in self.graph.derived_relations:
            if rel.relation_type == "POSSIBLY_EXCLUDED" and rel.value:
                pid_i = rel.participants[0]
                self.mskb.add(MentalStateEntry(
                    entry_id="", order=1, entry_type="FEELING",
                    subject=pid_i, predicate="may_feel_excluded_from_group",
                    value=True, confidence=rel.confidence * 0.6,
                    rule_applied="Rule 1.6: No One Gazing At + No Reciprocated Engagement → Possibly Excluded",
                    natural_language=f"P{pid_i} is not being looked at by anyone and has no reciprocated interactions — may feel excluded"
                ))

    # ---- 2阶推理 ----

    def _reason_order_2(self):
        pids = list(self.graph.nodes.keys())

        for entry in self.mskb.get_by_order(1):
            if entry.entry_type not in ["BELIEF", "IGNORANCE", "FEELING"]:
                continue

            subject = entry.subject

            for pid_k in pids:
                if pid_k == subject:
                    continue

                # Rule 2.1
                if entry.entry_type == "IGNORANCE" and entry.about is not None:
                    va_k_subject = self.graph.visual_access_matrix.get((pid_k, subject))
                    va_k_about = self.graph.visual_access_matrix.get((pid_k, entry.about))
                    va_subject_about = self.graph.visual_access_matrix.get((subject, entry.about))

                    if (va_k_subject and va_k_subject["can_observe"] and
                            va_k_about and va_k_about["can_observe"] and
                            va_subject_about and not va_subject_about["can_observe"]):
                        conf = (entry.confidence * self.confidence_decay *
                                min(va_k_subject["confidence"], va_k_about["confidence"]))
                        self.mskb.add(MentalStateEntry(
                            entry_id="", order=2, entry_type="RECURSIVE_BELIEF",
                            subject=pid_k,
                            predicate=f"knows_that_P{subject}_{entry.predicate}",
                            about=subject, value=True, confidence=conf,
                            rule_applied="Rule 2.1: Observer can see both parties → knows about information asymmetry",
                            premises=[entry.entry_id],
                            natural_language=f"P{pid_k} can observe both P{subject} and P{entry.about}, and can tell that P{subject} is unaware of P{entry.about}'s behavior"
                        ))

                # Rule 2.2
                if entry.entry_type == "BELIEF" and "aware_of_being_watched" in entry.predicate:
                    watcher = entry.about
                    va_k_subject = self.graph.visual_access_matrix.get((pid_k, subject))
                    if va_k_subject and va_k_subject["can_observe"]:
                        conf = entry.confidence * self.confidence_decay * va_k_subject["confidence"] * 0.7
                        self.mskb.add(MentalStateEntry(
                            entry_id="", order=2, entry_type="RECURSIVE_BELIEF",
                            subject=pid_k,
                            predicate=f"may_know_that_P{subject}_is_aware_of_P{watcher}_watching",
                            about=subject, value=True, confidence=conf,
                            rule_applied="Rule 2.2: Observer sees subject's awareness of being watched",
                            premises=[entry.entry_id],
                            natural_language=f"P{pid_k} can observe P{subject}, and may realize that P{subject} knows P{watcher} is watching"
                        ))

                # Rule 2.3
                if entry.entry_type == "FEELING":
                    va_k_subject = self.graph.visual_access_matrix.get((pid_k, subject))
                    if va_k_subject and va_k_subject["can_observe"]:
                        feel_part = entry.predicate.split('feel_')[-1] if 'feel_' in entry.predicate else 'something'
                        conf = entry.confidence * self.confidence_decay * va_k_subject["confidence"] * 0.6
                        self.mskb.add(MentalStateEntry(
                            entry_id="", order=2, entry_type="RECURSIVE_BELIEF",
                            subject=pid_k,
                            predicate=f"may_sense_that_P{subject}_feels_{feel_part}",
                            about=subject, value=True, confidence=conf,
                            rule_applied="Rule 2.3: Observer can see subject → may sense their feeling",
                            premises=[entry.entry_id],
                            natural_language=f"P{pid_k} can observe P{subject} and may sense that P{subject} {entry.predicate.replace('_', ' ')}"
                        ))

        # Rule 2.4: Secret Observer
        for pid_i in pids:
            target = self.graph.gazes_at.get(pid_i)
            if target is None:
                continue
            va_target_i = self.graph.visual_access_matrix.get((target, pid_i))
            if va_target_i and not va_target_i["can_observe"]:
                self.mskb.add(MentalStateEntry(
                    entry_id="", order=2, entry_type="RECURSIVE_BELIEF",
                    subject=pid_i,
                    predicate=f"knows_P{target}_is_unaware_of_being_watched",
                    about=target, value=True,
                    confidence=va_target_i["confidence"] * 0.8,
                    rule_applied="Rule 2.4: Secret Observer (I watch you, you can't see me)",
                    natural_language=f"P{pid_i} knows that P{target} is unaware of being watched by P{pid_i}, because P{target} cannot see P{pid_i}"
                ))

    # ---- 高阶推理 ----

    def _reason_higher_order(self, order: int) -> int:
        pids = list(self.graph.nodes.keys())
        prev_entries = self.mskb.get_by_order(order - 1)
        new_count = 0

        for entry in prev_entries:
            if entry.entry_type != "RECURSIVE_BELIEF":
                continue
            if entry.confidence < 0.3:
                continue

            subject = entry.subject
            for pid_k in pids:
                if pid_k == subject:
                    continue
                va_k_subject = self.graph.visual_access_matrix.get((pid_k, subject))
                if va_k_subject is None:
                    continue
                conf = entry.confidence * self.confidence_decay * va_k_subject["confidence"]
                if conf < 0.2:
                    continue
                if va_k_subject["can_observe"]:
                    self.mskb.add(MentalStateEntry(
                        entry_id="", order=order, entry_type="RECURSIVE_BELIEF",
                        subject=pid_k,
                        predicate=f"may_know_that_P{subject}_{entry.predicate}",
                        about=subject, value=True, confidence=conf,
                        rule_applied=f"Rule {order}.1: Higher-order recursive belief (order={order})",
                        premises=[entry.entry_id],
                        natural_language=f"P{pid_k} can observe P{subject}, and may realize that P{subject} {entry.predicate.replace('_', ' ')}"
                    ))
                    new_count += 1
        return new_count

    # ---- 社交动态推断 ----

    def _infer_social_dynamics(self):
        for entry in self.mskb.get_by_order(2):
            if "unaware" in entry.predicate and entry.value:
                self.mskb.add(MentalStateEntry(
                    entry_id="", order=0, entry_type="SOCIAL_DYNAMIC",
                    subject=entry.subject, predicate="has_information_advantage",
                    about=entry.about, value=True, confidence=entry.confidence,
                    rule_applied="Dynamic: Information Asymmetry Detection",
                    premises=[entry.entry_id],
                    natural_language=f"P{entry.subject} has an information advantage over P{entry.about} — P{entry.subject} knows something that P{entry.about} doesn't know"
                ))

        for rel in self.graph.derived_relations:
            if rel.relation_type == "ATTENTION_CENTER":
                center_pid = rel.participants[0]
                self.mskb.add(MentalStateEntry(
                    entry_id="", order=0, entry_type="SOCIAL_DYNAMIC",
                    subject=center_pid, predicate="is_attention_center",
                    value=True, confidence=rel.confidence,
                    rule_applied="Dynamic: Attention Center Detection",
                    natural_language=f"P{center_pid} is the center of attention — being looked at by multiple people"
                ))

            if rel.relation_type == "POSSIBLY_EXCLUDED":
                iso_pid = rel.participants[0]
                self.mskb.add(MentalStateEntry(
                    entry_id="", order=0, entry_type="SOCIAL_DYNAMIC",
                    subject=iso_pid, predicate="is_possibly_isolated",
                    value=True, confidence=rel.confidence,
                    rule_applied="Dynamic: Social Isolation Detection",
                    natural_language=f"P{iso_pid} appears socially isolated — not being engaged by others"
                ))


# ================================================================
#               MODULE D: LLM-Grounded Reasoning
# ================================================================

class LLMReasoningModule:
    """LLM辅助推理与问答"""

    def __init__(self, model_name: str = MODEL_NAME, max_retries: int = 3):
        self.model_name = model_name
        self.max_retries = max_retries

    def _encode_image(self, pil_image: Image.Image,
                      max_size: int = 512) -> str:
        w, h = pil_image.size
        if max(w, h) > max_size:
            ratio = max_size / max(w, h)
            pil_image = pil_image.resize(
                (int(w * ratio), int(h * ratio)), Image.LANCZOS
            )
        buffer = BytesIO()
        pil_image.save(buffer, format="JPEG", quality=85)
        return base64.b64encode(buffer.getvalue()).decode("utf-8")

    def _call_llm(self, messages: list, temperature: float = 0.3,
                  max_tokens: int = 2000) -> str:
        for attempt in range(self.max_retries):
            try:
                response = client.chat.completions.create(
                    model=self.model_name,
                    messages=messages,
                    temperature=temperature,
                    max_tokens=max_tokens
                )
                return response.choices[0].message.content
            except Exception as e:
                print(f"  LLM API call failed (attempt {attempt + 1}): {e}")
                if attempt < self.max_retries - 1:
                    time.sleep(2 ** attempt)
                else:
                    return f"[LLM Error: {str(e)}]"

    def _call_llm_with_image(self, system_prompt: str,
                             user_prompt: str,
                             pil_image: Image.Image,
                             temperature: float = 0.3,
                             max_tokens: int = 2000) -> str:
        img_b64 = self._encode_image(pil_image)
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": [
                {"type": "image_url", "image_url": {
                    "url": f"data:image/jpeg;base64,{img_b64}"
                }},
                {"type": "text", "text": user_prompt}
            ]}
        ]
        return self._call_llm(messages, temperature, max_tokens)

    # ---- D1: 模糊推理补充 ----

    def complementary_reasoning(self,
                                mskb: MentalStateKnowledgeBase,
                                graph: SocialMindGraph,
                                pil_image: Image.Image) -> str:
        system_prompt = """You are a social cognition expert analyzing multi-party social scenes.

You will be given:
1. An image of a social scene
2. Formally verified mental states (derived by symbolic reasoning from visual social signals like gaze direction)
3. The Social Mind Graph description

Your task: Provide complementary reasoning that the symbolic rules cannot cover.
Focus on:
- Scene context (what kind of social setting is this?)
- Specific intentions (why might someone be behaving this way?)
- Social dynamics that require world knowledge

CRITICAL RULES:
- Do NOT contradict any verified facts
- Clearly label your inferences as [INFERRED] vs verified facts as [VERIFIED]
- Express uncertainty when appropriate
- Be concise and structured"""

        user_prompt = f"""
=== SOCIAL MIND GRAPH ===
{graph.to_natural_language()}

=== VERIFIED MENTAL STATES (from symbolic reasoning) ===
{mskb.to_natural_language()}

=== YOUR TASK ===
Based on the image and the verified facts above, provide:

1. SCENE CONTEXT: What kind of social setting is this? (e.g., classroom, meeting, casual conversation)

2. CONTEXTUAL INTENT: For each person, what might be their specific social goal given the context?

3. UNRESOLVED QUESTIONS: What aspects of the social dynamics cannot be determined from gaze alone and would require additional information (audio, facial expression details, etc.)?

4. OVERALL SOCIAL DYNAMIC SUMMARY: In 2-3 sentences, describe the social dynamic of this scene, integrating both verified facts and your contextual inferences.
"""
        return self._call_llm_with_image(system_prompt, user_prompt, pil_image)

    # ---- D2: Query Answering ----

    def answer_query(self,
                     query: str,
                     mskb: MentalStateKnowledgeBase,
                     graph: SocialMindGraph,
                     pil_image: Image.Image,
                     complementary_analysis: str = "") -> dict:
        system_prompt = """You are an expert social scene analyst using a neuro-symbolic reasoning framework.

You will be given:
1. An image of a multi-party social scene
2. A Social Mind Graph (who can see whom, who is looking at whom)
3. Formally VERIFIED mental states derived by symbolic Theory of Mind rules
4. Contextual analysis from a complementary reasoning module
5. A specific question to answer

YOUR RESPONSE FORMAT (strictly follow this JSON structure):
{
    "answer": "Your answer to the question",
    "evidence_chain": [
        {"type": "verified", "fact": "description of verified fact", "rule": "which rule produced this"},
        {"type": "inferred", "fact": "description of inferred fact", "basis": "what this is based on"}
    ],
    "confidence": "high/medium/low",
    "tom_order_used": 0,
    "reasoning": "Step-by-step reasoning trace"
}

CRITICAL RULES:
- Always ground your answer in VERIFIED facts when possible
- Clearly distinguish [VERIFIED by symbolic reasoning] vs [INFERRED by contextual reasoning]
- Report the highest ToM order used in your reasoning (0=observation, 1=belief/intent, 2=recursive belief, 3+=higher)
- If the question cannot be answered from available evidence, say so honestly
- Consider what each person CAN and CANNOT see (Visual Access Matrix) — this is crucial for ToM questions"""

        user_prompt = f"""
=== SOCIAL MIND GRAPH ===
{graph.to_natural_language()}

=== VERIFIED MENTAL STATES ===
{mskb.to_natural_language()}

=== CONTEXTUAL ANALYSIS ===
{complementary_analysis if complementary_analysis else "(not yet performed)"}

=== QUESTION ===
{query}

Please answer the question following the required JSON format. Ground your answer in the verified facts and visual evidence.
"""
        raw_response = self._call_llm_with_image(
            system_prompt, user_prompt, pil_image,
            temperature=0.2, max_tokens=1500
        )
        return self._parse_response(raw_response, query)

    def _parse_response(self, raw_response: str, query: str) -> dict:
        try:
            json_match = re.search(r'\{[\s\S]*\}', raw_response)
            if json_match:
                result = json.loads(json_match.group())
                result.setdefault("answer", "")
                result.setdefault("evidence_chain", [])
                result.setdefault("confidence", "medium")
                result.setdefault("tom_order_used", 0)
                result.setdefault("reasoning", "")
                return result
        except json.JSONDecodeError:
            pass

        return {
            "answer": raw_response,
            "evidence_chain": [],
            "confidence": "unknown",
            "tom_order_used": -1,
            "reasoning": "Failed to parse structured response"
        }

    # ---- D3: Social-IQ Format QA ----

    def answer_social_iq(self,
                         question: str,
                         answer_choices: List[dict],
                         mskb: MentalStateKnowledgeBase,
                         graph: SocialMindGraph,
                         pil_image: Image.Image,
                         complementary_analysis: str = "") -> dict:
        system_prompt = """You are an expert social scene analyst using a neuro-symbolic Theory of Mind framework.

You will be given a multi-party social scene with:
1. Visual evidence (image)
2. Formally verified mental states from symbolic ToM reasoning
3. A multiple-choice question about social dynamics

Your task: Select the best answer and explain your reasoning.

RESPONSE FORMAT (JSON):
{
    "selected_answer_index": 0,
    "selected_answer_text": "the answer text",
    "reasoning": "step-by-step reasoning grounded in verified facts",
    "evidence_used": ["list of key verified facts used"],
    "tom_order_required": 0,
    "confidence": "high/medium/low"
}

RULES:
- Ground reasoning in VERIFIED mental states when relevant
- Consider Visual Access (who can see whom) for perspective-taking questions
- Analyze each answer choice independently based on the evidence; do not assume a specific option index is inherently correct or incorrect."""

        choices_text = "\n".join([
            f"  [{c.get('index', i)}] {c['text']}"
            for i, c in enumerate(answer_choices)
        ])

        verified_facts = mskb.to_natural_language(min_confidence=0.3)
        graph_desc = graph.to_natural_language()

        user_prompt = f"""
=== SOCIAL MIND GRAPH ===
{graph_desc}

=== VERIFIED MENTAL STATES ===
{verified_facts}

=== CONTEXTUAL ANALYSIS ===
{complementary_analysis if complementary_analysis else "(not available)"}

=== QUESTION ===
{question}

=== ANSWER CHOICES ===
{choices_text}

Select the best answer (by index) and explain your reasoning.
"""
        raw_response = self._call_llm_with_image(
            system_prompt, user_prompt, pil_image,
            temperature=0.1, max_tokens=1500
        )
        result = self._parse_response(raw_response, question)
        result["question"] = question
        result["choices"] = answer_choices
        return result


# ================================================================
#          Social-IQ 数据解析器
# ================================================================

class SocialIQParser:
    """解析Social-IQ数据集的QA文件"""

    @staticmethod
    def parse_qa_file(filepath: str, target_vid_name: str = None) -> List[dict]:
        """
        解析Social-IQ 2.0 的 JSON/JSONL QA文件

        Args:
            filepath: json/jsonl 文件的路径
            target_vid_name: 可选。如果传入的是一个包含所有视频的大json文件，
                             可以通过该参数只筛选出当前测试视频的 QA。
        """
        questions = []

        with open(filepath, 'r', encoding='utf-8') as f:
            content = f.read().strip()
            if content.startswith('['):
                data_list = json.loads(content)
            else:
                data_list = [json.loads(line) for line in content.split('\n') if line.strip()]

        for data in data_list:
            if target_vid_name and data.get("vid_name") != target_vid_name:
                continue

            q_id = data.get("qid")
            q_text = data.get("q")
            correct_idx = data.get("answer_idx", -1)

            current_q = {
                "question_id": q_id,
                "question": q_text,
                "vid_name": data.get("vid_name"),
                "answers": [],
                "correct_indices": [],
                "incorrect_indices": []
            }

            for i in range(4):
                ans_key = f"a{i}"
                if ans_key in data:
                    text = data[ans_key]
                    is_correct = (i == correct_idx)
                    label = 'a' if is_correct else 'i'

                    current_q["answers"].append({
                        "index": i,
                        "label": label,
                        "text": text,
                        "is_correct": is_correct
                    })
                    if is_correct:
                        current_q["correct_indices"].append(i)
                    else:
                        current_q["incorrect_indices"].append(i)

            questions.append(current_q)

        return questions


# ================================================================
#          完整Pipeline整合
# ================================================================

class SocialMindPipeline:
    """
    完整的Neuro-Symbolic Social Mind Reasoning Pipeline

    Module A (Neural Perception) → Module B (Graph) →
    Module C (Symbolic ToM) → Module D (LLM Reasoning)
    """

    def __init__(self,
                 gazelle_model, gazelle_transform, device,
                 face_detector,
                 max_persons=5, inout_thresh=0.5, yolo_conf=0.7,
                 max_tom_order=3):

        # Module A
        self.tracker = PersonTracker(
            iou_threshold=0.3, max_disappear=30, max_persons=max_persons
        )
        self.gaze_detector = GazeDetector(
            model=gazelle_model, transform=gazelle_transform,
            device=device, inout_threshold=inout_thresh
        )
        self.yolo = face_detector
        self.yolo_conf = yolo_conf
        self.max_persons = max_persons

        # Module B
        self.graph = SocialMindGraph(fov_angle_deg=150)

        # Module C
        self.tom_engine = SymbolicToMEngine(max_order=max_tom_order)

        # Module D
        self.llm_module = LLMReasoningModule()

        # 存储
        self.frame_results = []

    def process_video_and_answer(self,
                                 video_path: str,
                                 qa_file_path: str,
                                 output_video_path: str = None,
                                 output_json_path: str = None,
                                 sample_frames: List[int] = None,
                                 keyframe_interval: int = 30) -> dict:
        """
        完整Pipeline：处理视频 + 回答Social-IQ问题
        """

        # 1. 解析QA文件
        print("=" * 60)
        print("STEP 1: Parsing Social-IQ QA file")
        print("=" * 60)

        vid_name = os.path.splitext(os.path.basename(video_path))[0]
        qa_data = SocialIQParser.parse_qa_file(qa_file_path, target_vid_name=vid_name)
        print(f"  Loaded {len(qa_data)} questions")
        for q in qa_data:
            print(f"    {q['question_id']}: {q['question'][:60]}...")

        # 2. 处理视频
        print("\n" + "=" * 60)
        print("STEP 2: Processing video (Module A + B + C)")
        print("=" * 60)

        keyframe_data = self._process_video_collect_keyframes(
            video_path, output_video_path,
            sample_frames, keyframe_interval
        )
        print(f"\n  Collected {len(keyframe_data)} keyframes for analysis")

        # 3. 选择最具代表性的关键帧
        print("\n" + "=" * 60)
        print("STEP 3: Selecting representative keyframe")
        print("=" * 60)

        best_frame = self._select_best_keyframe(keyframe_data)

        if best_frame is None:
            print("  WARNING: No valid keyframe found!")
            return {"error": "No valid keyframes with detected persons"}

        print(f"  Selected frame {best_frame['frame_idx']} "
              f"({best_frame['num_persons']} persons, "
              f"{best_frame['num_interactions']} interactions)")

        # 4. Module D
        print("\n" + "=" * 60)
        print("STEP 4: LLM-Grounded Reasoning (Module D)")
        print("=" * 60)

        pil_image = best_frame["pil_image"]
        mskb = best_frame["mskb"]
        graph = best_frame["graph"]

        # D1
        print("\n  [D1] Running complementary reasoning...")
        complementary = self.llm_module.complementary_reasoning(
            mskb, graph, pil_image
        )
        print(f"  Complementary analysis:\n{complementary[:500]}...")

        # D2/D3
        print("\n  [D2] Answering Social-IQ questions...")

        all_qa_results = []
        for q_data in qa_data:
            print(f"\n  --- {q_data['question_id']}: {q_data['question'][:60]}... ---")

            result = self.llm_module.answer_social_iq(
                question=q_data["question"],
                answer_choices=q_data["answers"],
                mskb=mskb,
                graph=graph,
                pil_image=pil_image,
                complementary_analysis=complementary
            )

            if "selected_answer_index" in result:
                idx = result.get("selected_answer_index", -1)
                print(f"  Selected: [{idx}] {result.get('selected_answer_text', 'N/A')}")
            print(f"  Confidence: {result.get('confidence', 'N/A')}")
            print(f"  ToM order: {result.get('tom_order_required', 'N/A')}")
            reasoning = result.get('reasoning', '')
            if reasoning:
                print(f"  Reasoning: {reasoning[:200]}...")

            all_qa_results.append({
                "question_id": q_data["question_id"],
                "question": q_data["question"],
                "ground_truth_answers": q_data["answers"],
                "model_response": result
            })

        # 5. 汇总
        print("\n" + "=" * 60)
        print("STEP 5: Compiling results")
        print("=" * 60)

        full_result = {
            "video_path": video_path,
            "qa_file": qa_file_path,
            "keyframe_info": {
                "frame_idx": best_frame["frame_idx"],
                "num_persons": best_frame["num_persons"],
                "num_interactions": best_frame["num_interactions"]
            },
            "social_mind_graph": graph.to_dict(),
            "mental_state_knowledge_base": mskb.to_dict(),
            "complementary_analysis": complementary,
            "qa_results": all_qa_results
        }

        if output_json_path:
            with open(output_json_path, 'w', encoding='utf-8') as f:
                json.dump(full_result, f, indent=2, ensure_ascii=False, default=str)
            print(f"\n  Results saved to: {output_json_path}")

        self._print_summary(full_result)
        return full_result

    def _process_video_collect_keyframes(self,
                                         video_path: str,
                                         output_video_path: str = None,
                                         sample_frames: List[int] = None,
                                         keyframe_interval: int = 30) -> List[dict]:
        """处理视频并收集关键帧数据"""

        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            raise ValueError(f"Cannot open video: {video_path}")

        fps = int(cap.get(cv2.CAP_PROP_FPS))
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        print(f"  Video: {width}x{height}, {fps}FPS, {total_frames} frames")

        out_writer = None
        if output_video_path:
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            out_writer = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

        if sample_frames is None:
            sample_frames = list(range(0, total_frames, keyframe_interval))
        sample_set = set(sample_frames)

        keyframe_data = []
        frame_count = 0

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            frame_count += 1
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_image = Image.fromarray(rgb_frame)

            # Module A1
            results = self.yolo(rgb_frame, verbose=False, conf=self.yolo_conf)
            detected_boxes, detected_confs = [], []
            if len(results) > 0 and len(results[0].boxes) > 0:
                boxes_np = results[0].boxes.xyxy.cpu().numpy()
                confs_np = results[0].boxes.conf.cpu().numpy()
                areas = (boxes_np[:, 2] - boxes_np[:, 0]) * (boxes_np[:, 3] - boxes_np[:, 1])
                sorted_idx = np.argsort(areas)[::-1][:self.max_persons]
                for idx in sorted_idx:
                    detected_boxes.append(boxes_np[idx].tolist())
                    detected_confs.append(float(confs_np[idx]))

            active_persons = self.tracker.update(
                detected_boxes, detected_confs, frame_count, width, height
            )

            # Module A2
            if len(active_persons) > 0:
                active_persons = self.gaze_detector.detect(pil_image, active_persons)

            # Module B
            self.graph.build(active_persons, width, height)

            # 关键帧处理: Module C
            if frame_count in sample_set and len(active_persons) >= 2:
                mskb = self.tom_engine.reason(self.graph)
                num_interactions = len([e for e in self.graph.edges if e.edge_type == "GAZES_AT"])

                keyframe_data.append({
                    "frame_idx": frame_count,
                    "pil_image": pil_image.copy(),
                    "active_persons": dict(active_persons),
                    "graph": self.graph,
                    "graph_dict": self.graph.to_dict(),
                    "graph_nl": self.graph.to_natural_language(),
                    "mskb": mskb,
                    "mskb_dict": mskb.to_dict(),
                    "mskb_nl": mskb.to_natural_language(),
                    "num_persons": len(active_persons),
                    "num_interactions": num_interactions
                })
                print(f"  [Keyframe {frame_count}] "
                      f"{len(active_persons)} persons, "
                      f"{num_interactions} gaze interactions")

            # 可视化输出
            if out_writer and len(active_persons) > 0:
                vis_image = visualize_with_ids(pil_image, active_persons, self.graph)
                out_writer.write(cv2.cvtColor(np.array(vis_image), cv2.COLOR_RGB2BGR))
            elif out_writer:
                out_writer.write(frame)

            if frame_count % (fps * 5) == 0:
                print(f"  Processing: {frame_count}/{total_frames} frames")

        cap.release()
        if out_writer:
            out_writer.release()
            print(f"  Output video saved: {output_video_path}")

        return keyframe_data

    def _select_best_keyframe(self, keyframe_data: List[dict]) -> Optional[dict]:
        """选择最具代表性的关键帧"""
        if not keyframe_data:
            return None

        scored = [(kf, kf["num_persons"] * 2 + kf["num_interactions"] * 3)
                  for kf in keyframe_data]
        scored.sort(key=lambda x: x[1], reverse=True)
        best = scored[0][0]

        # 重建 graph 和 mskb（graph 对象会被后续帧覆盖）
        pil_image = best["pil_image"]
        rgb_frame = np.array(pil_image)

        results = self.yolo(rgb_frame, verbose=False, conf=self.yolo_conf)
        detected_boxes, detected_confs = [], []
        if len(results) > 0 and len(results[0].boxes) > 0:
            boxes_np = results[0].boxes.xyxy.cpu().numpy()
            confs_np = results[0].boxes.conf.cpu().numpy()
            areas = (boxes_np[:, 2] - boxes_np[:, 0]) * (boxes_np[:, 3] - boxes_np[:, 1])
            sorted_idx = np.argsort(areas)[::-1][:self.max_persons]
            for idx in sorted_idx:
                detected_boxes.append(boxes_np[idx].tolist())
                detected_confs.append(float(confs_np[idx]))

        w, h = pil_image.size
        temp_tracker = PersonTracker(max_persons=self.max_persons)
        active_persons = temp_tracker.update(detected_boxes, detected_confs, 1, w, h)

        if len(active_persons) > 0:
            active_persons = self.gaze_detector.detect(pil_image, active_persons)

        fresh_graph = SocialMindGraph(fov_angle_deg=150)
        fresh_graph.build(active_persons, w, h)

        fresh_engine = SymbolicToMEngine(max_order=3)
        fresh_mskb = fresh_engine.reason(fresh_graph)

        best["graph"] = fresh_graph
        best["mskb"] = fresh_mskb
        best["graph_nl"] = fresh_graph.to_natural_language()
        best["mskb_nl"] = fresh_mskb.to_natural_language()

        return best

    def _print_summary(self, result: dict):
        """打印最终摘要"""
        print("\n" + "=" * 60)
        print("  PIPELINE EXECUTION SUMMARY")
        print("=" * 60)
        kf = result["keyframe_info"]
        print(f"\n  Keyframe: #{kf['frame_idx']} "
              f"({kf['num_persons']} persons, {kf['num_interactions']} interactions)")

        graph_summary = result["social_mind_graph"].get("summary", {})
        print(f"\n  Gaze interactions: {graph_summary.get('gaze_interactions', [])}")
        print(f"  Mutual gaze: {graph_summary.get('mutual_gaze_pairs', [])}")

        mskb_data = result["mental_state_knowledge_base"]
        for order_key in sorted(mskb_data.keys()):
            entries = mskb_data[order_key]
            print(f"  {order_key}: {len(entries)} mental state entries")

        print(f"\n  Questions answered: {len(result['qa_results'])}")
        correct_count = 0
        for qa in result["qa_results"]:
            q_id = qa["question_id"]
            resp = qa["model_response"]
            conf = resp.get("confidence", "N/A")
            tom = resp.get("tom_order_required", "N/A")
            answer = resp.get("answer", resp.get("selected_answer_text", "N/A"))

            pred_idx = resp.get("selected_answer_index", -1)
            gt_answers = qa.get("ground_truth_answers", [])
            is_correct = any(
                ans.get("index") == pred_idx and ans.get("is_correct")
                for ans in gt_answers
            )
            if is_correct:
                correct_count += 1
            mark = "✓" if is_correct else "✗"
            print(f"    {mark} {q_id}: [conf={conf}, ToM={tom}] {str(answer)[:80]}...")

        total_q = len(result['qa_results'])
        if total_q > 0:
            acc = correct_count / total_q * 100
            print(f"\n  ====================================")
            print(f"  Accuracy: {correct_count}/{total_q} ({acc:.1f}%)")
            print(f"  ====================================")


# ================================================================
#          独立视频处理（A+B可视化，不调用LLM）
# ================================================================

def process_video_full_pipeline(video_path: str, output_path: str,
                                json_output_path: str = None,
                                max_persons: int = 5,
                                inout_thresh: float = 0.5,
                                yolo_conf: float = 0.7):
    """
    仅Module A1 + A2 + Module B 的可视化Pipeline（不调用LLM）
    """
    tracker = PersonTracker(iou_threshold=0.3, max_disappear=30, max_persons=max_persons)
    gaze_detector = GazeDetector(
        model=gazelle_model, transform=gazelle_transform,
        device=device, inout_threshold=inout_thresh,
        gaze_to_person_radius=0.15
    )
    graph = SocialMindGraph(fov_angle_deg=150)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"无法打开视频: {video_path}")
        return

    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"视频信息: {width}x{height}, {fps} FPS, 总帧数: {total_frames}")

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    all_frame_graphs = []
    frame_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_count += 1
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_image = Image.fromarray(rgb_frame)

        # Module A1
        results = face_detector(rgb_frame, verbose=False, conf=yolo_conf)
        detected_boxes, detected_confs = [], []
        if len(results) > 0 and len(results[0].boxes) > 0:
            boxes_tensor = results[0].boxes.xyxy.cpu().numpy()
            confs_tensor = results[0].boxes.conf.cpu().numpy()
            areas = (boxes_tensor[:, 2] - boxes_tensor[:, 0]) * \
                    (boxes_tensor[:, 3] - boxes_tensor[:, 1])
            sorted_idx = np.argsort(areas)[::-1][:max_persons]
            for idx in sorted_idx:
                detected_boxes.append(boxes_tensor[idx].tolist())
                detected_confs.append(float(confs_tensor[idx]))

        active_persons = tracker.update(
            detected_boxes, detected_confs, frame_count, width, height
        )

        # Module A2
        if len(active_persons) > 0:
            active_persons = gaze_detector.detect(pil_image, active_persons)

        # Module B
        graph.build(active_persons, width, height)

        all_frame_graphs.append({"frame": frame_count, "graph": graph.to_dict()})

        if len(active_persons) > 0:
            vis_image = visualize_with_ids(pil_image, active_persons, graph, inout_thresh)
            final_frame = cv2.cvtColor(np.array(vis_image), cv2.COLOR_RGB2BGR)
        else:
            final_frame = frame

        out.write(final_frame)

        if frame_count % 30 == 0:
            print(f"\n[Frame {frame_count}/{total_frames}]")
            print(f"  Active persons: {list(active_persons.keys())}")
            summary = graph.to_dict().get("summary", {})
            if summary:
                print(f"  Gaze interactions: {summary.get('gaze_interactions', [])}")
                print(f"  Mutual gaze: {summary.get('mutual_gaze_pairs', [])}")
            if frame_count % 150 == 0:
                print(graph.to_natural_language())

    cap.release()
    out.release()

    if json_output_path:
        sampled = [fg for fg in all_frame_graphs if fg["frame"] % fps == 0]
        with open(json_output_path, 'w', encoding='utf-8') as f:
            json.dump(sampled, f, indent=2, default=str)
        print(f"Graph数据保存至: {json_output_path} ({len(sampled)} frames)")

    print(f"\n完成! 视频保存至: {output_path}")
    print(f"总处理帧数: {frame_count}")
    print(f"分配的Person ID数: {tracker.next_id - 1}")


# ================================================================
#                          运行
# ================================================================

if __name__ == "__main__":

    # ---- 路径配置 ----
    video_path = "/Users/zehao/Desktop/HAI/MultiParty/Social-IQ-2/siq2/video/cGU1Pepn1hU.mp4"
    qa_file_path = "/Users/zehao/Desktop/HAI/MultiParty/Social-IQ-2/siq2/qa/qa_train.json"
    video_output_path = "/Users/zehao/Desktop/HAI/MultiParty/code/output_full_pipeline.mp4"
    json_output_path = "/Users/zehao/Desktop/HAI/MultiParty/code/output_full_results.json"

    os.makedirs(os.path.dirname(video_output_path), exist_ok=True)

    # ---- 初始化完整Pipeline ----
    pipeline = SocialMindPipeline(
        gazelle_model=gazelle_model,
        gazelle_transform=gazelle_transform,
        device=device,
        face_detector=face_detector,
        max_persons=5,
        inout_thresh=0.5,
        yolo_conf=0.7,
        max_tom_order=3
    )

    # ---- 运行完整Pipeline ----
    results = pipeline.process_video_and_answer(
        video_path=video_path,
        qa_file_path=qa_file_path,
        output_video_path=video_output_path,
        output_json_path=json_output_path,
        keyframe_interval=30
    )

    # ---- 打印完整的MSKB ----
    print("\n\n" + "=" * 60)
    print("  FULL MENTAL STATE ANALYSIS")
    print("=" * 60)

    mskb_data = results.get("mental_state_knowledge_base", {})
    for order_key in sorted(mskb_data.keys()):
        print(f"\n--- {order_key} ---")
        for entry in mskb_data[order_key]:
            print(f"  [{entry['type']}] {entry['natural_language']}")
            print(f"    Rule: {entry['rule']}")
            print(f"    Confidence: {entry['confidence']}")

Using device: cpu


Using cache found in /Users/zehao/Desktop/HAI/MultiParty/code/gazelle-main/fkryan_gazelle_main
/opt/miniconda3/envs/siq2/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using cache found in /Users/zehao/Desktop/HAI/MultiParty/code/gazelle-main/facebookresearch_dinov2_main
/Users/zehao/Desktop/HAI/MultiParty/code/gazelle-main/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:45: UserWarning: xFormers is disabled (SwiGLU)
  warnings.warn("xFormers is disabled (SwiGLU)")
/Users/zehao/Desktop/HAI/MultiParty/code/gazelle-main/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/Users/zehao/Desktop/HAI/MultiParty/code/gazelle-main/facebookresearch_dinov2_main/dinov2/layers/attention.py:29: User

STEP 1: Parsing Social-IQ QA file
  Loaded 5 questions
    cGU1Pepn1hU_q1_3: Do the people dislike math?...
    cGU1Pepn1hU_q4_1: Why are the people clapping?...
    cGU1Pepn1hU_q2_9: Why does the crowd clap each time after the man finishes tal...
    cGU1Pepn1hU_q5_6: Why does the man in grey think he was lucky?...
    cGU1Pepn1hU_q3_9: Why does the man in the center of the video with his back tu...

STEP 2: Processing video (Module A + B + C)
  Video: 640x360, 29FPS, 1799 frames
  [Keyframe 30] 2 persons, 0 gaze interactions
  [Keyframe 90] 2 persons, 1 gaze interactions
  Processing: 145/1799 frames
  Processing: 290/1799 frames
  [Keyframe 390] 2 persons, 1 gaze interactions
  [Keyframe 420] 2 persons, 1 gaze interactions
  Processing: 435/1799 frames
  [Keyframe 450] 3 persons, 1 gaze interactions
  [Keyframe 480] 3 persons, 1 gaze interactions
  Processing: 580/1799 frames
  [Keyframe 600] 2 persons, 1 gaze interactions
  [Keyframe 630] 3 persons, 1 gaze interactions
  Processing

In [ ]:

"""
Neuro-Symbolic Social Mind Reasoning Pipeline - Dataset Evaluation Mode
"""
 
import os
import cv2
import torch
import numpy as np
import json
import math
import re
import time
import base64
from io import BytesIO
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple, Any, Set
from collections import defaultdict
from PIL import Image, ImageDraw, ImageFont
 
from ultralytics import YOLO
import openai
 
# ================= 环境设置 =================
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ["XFORMERS_DISABLED"] = "1"
 
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
 
# ================= API 配置 =================
API_KEY = "sk-bbf9f24ff4194920a43e15749a2dad29"
API_BASE = "https://dashscope.aliyuncs.com/compatible-mode/v1"
MODEL_NAME = "qwen3.6-plus"
 
client = openai.OpenAI(api_key=API_KEY, base_url=API_BASE)
 
# ================= 模型加载 =================
torch.hub.set_dir(r'/Users/zehao/Desktop/HAI/MultiParty/code/gazelle-main')
gazelle_model, gazelle_transform = torch.hub.load(
    'fkryan/gazelle', 'gazelle_dinov2_vitb14_inout'
)
gazelle_model.eval()
gazelle_model.to(device)
 
face_detector = YOLO('yolov8n-face.pt')
 
# ================================================================
#                    MODULE A1 & A2 & B & C
# (与原代码完全一致，为节省篇幅省略重复注释)
# ================================================================
 
@dataclass
class TrackedPerson:
    person_id: int
    bbox: List[float]
    bbox_norm: List[float]
    confidence: float
    last_seen_frame: int
    gaze_target_pos: Optional[Tuple[float, float]] = None
    gaze_target_person: Optional[int] = None
    gaze_target_type: str = "unknown"
    gaze_heatmap: Optional[np.ndarray] = None
    inout_score: float = 0.0
    head_direction: Optional[Tuple[float, float]] = None
    gaze_confidence: float = 0.0
 
class PersonTracker:
    def __init__(self, iou_threshold=0.3, max_disappear=30, max_persons=10):
        self.iou_threshold = iou_threshold
        self.max_disappear = max_disappear
        self.max_persons = max_persons
        self.next_id = 1
        self.tracked: Dict[int, TrackedPerson] = {}
        self.current_frame = 0
 
    def reset(self):
        self.next_id = 1
        self.tracked.clear()
        self.current_frame = 0
 
    @staticmethod
    def compute_iou(box1, box2):
        x1, y1 = max(box1[0], box2[0]), max(box1[1], box2[1])
        x2, y2 = min(box1[2], box2[2]), min(box1[3], box2[3])
        inter = max(0, x2 - x1) * max(0, y2 - y1)
        area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
        area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
        union = area1 + area2 - inter
        return inter / union if union > 0 else 0
 
    def update(self, detections, confidences, frame_idx, img_width, img_height):
        self.current_frame = frame_idx
        active_ids = list(self.tracked.keys())
        active_boxes = [self.tracked[pid].bbox for pid in active_ids]
        matched_det, matched_track = set(), set()
 
        if len(active_ids) > 0 and len(detections) > 0:
            iou_matrix = np.zeros((len(active_ids), len(detections)))
            for i, track_box in enumerate(active_boxes):
                for j, det_box in enumerate(detections):
                    iou_matrix[i, j] = self.compute_iou(track_box, det_box)
            while True:
                if iou_matrix.size == 0: break
                max_iou = np.max(iou_matrix)
                if max_iou < self.iou_threshold: break
                i, j = np.unravel_index(np.argmax(iou_matrix), iou_matrix.shape)
                pid = active_ids[i]
                self.tracked[pid].bbox = detections[j]
                self.tracked[pid].bbox_norm = [detections[j][0]/img_width, detections[j][1]/img_height, detections[j][2]/img_width, detections[j][3]/img_height]
                self.tracked[pid].confidence = confidences[j]
                self.tracked[pid].last_seen_frame = frame_idx
                self._reset_gaze(pid)
                matched_det.add(j); matched_track.add(i)
                iou_matrix[i, :] = -1; iou_matrix[:, j] = -1
 
        for j in range(len(detections)):
            if j not in matched_det and len(self.tracked) < self.max_persons:
                pid = self.next_id; self.next_id += 1
                self.tracked[pid] = TrackedPerson(pid, detections[j], [detections[j][0]/img_width, detections[j][1]/img_height, detections[j][2]/img_width, detections[j][3]/img_height], confidences[j], frame_idx)
 
        to_remove = [pid for pid, p in self.tracked.items() if frame_idx - p.last_seen_frame > self.max_disappear]
        for pid in to_remove: del self.tracked[pid]
        return {pid: p for pid, p in self.tracked.items() if p.last_seen_frame == frame_idx}
 
    def _reset_gaze(self, pid):
        p = self.tracked[pid]
        p.gaze_target_pos = None; p.gaze_target_person = None; p.gaze_target_type = "unknown"
        p.gaze_heatmap = None; p.inout_score = 0.0; p.head_direction = None; p.gaze_confidence = 0.0
 
class GazeDetector:
    def __init__(self, model, transform, device, inout_threshold=0.5, gaze_to_person_radius=0.15):
        self.model = model; self.transform = transform; self.device = device
        self.inout_threshold = inout_threshold; self.gaze_to_person_radius = gaze_to_person_radius
 
    def detect(self, pil_image, active_persons):
        if len(active_persons) == 0: return active_persons
        img_tensor = self.transform(pil_image).unsqueeze(0).to(self.device)
        pid_list = list(active_persons.keys())
        norm_bboxes_np = [np.array([active_persons[pid].bbox_norm for pid in pid_list])]
        with torch.no_grad():
            output = self.model({"images": img_tensor, "bboxes": norm_bboxes_np})
        heatmaps = output['heatmap'][0]
        inout_scores = output['inout'][0] if output['inout'] is not None else None
 
        for i, pid in enumerate(pid_list):
            person = active_persons[pid]
            heatmap = heatmaps[i].detach().cpu().numpy()
            inout = inout_scores[i].item() if inout_scores is not None else 1.0
            person.inout_score = inout; person.gaze_heatmap = heatmap
            if inout > self.inout_threshold:
                max_idx = np.unravel_index(np.argmax(heatmap), heatmap.shape)
                gaze_y, gaze_x = max_idx[0] / heatmap.shape[0], max_idx[1] / heatmap.shape[1]
                person.gaze_target_pos = (gaze_x, gaze_y); person.gaze_confidence = float(np.max(heatmap))
                face_cx, face_cy = (person.bbox_norm[0]+person.bbox_norm[2])/2, (person.bbox_norm[1]+person.bbox_norm[3])/2
                dx, dy = gaze_x - face_cx, gaze_y - face_cy
                norm = math.sqrt(dx**2 + dy**2) + 1e-8
                person.head_direction = (dx/norm, dy/norm)
                person.gaze_target_person, person.gaze_target_type = self._map_gaze_to_person(gaze_x, gaze_y, pid, active_persons)
            else:
                person.gaze_target_type = "out_of_frame"; person.gaze_confidence = inout
        return active_persons
 
    def _map_gaze_to_person(self, gaze_x, gaze_y, self_pid, active_persons):
        best_pid, best_dist = None, float('inf')
        for pid, person in active_persons.items():
            if pid == self_pid: continue
            bx1, by1, bx2, by2 = person.bbox_norm
            cx, cy = (bx1+bx2)/2, (by1+by2)/2
            dist = math.sqrt((gaze_x-cx)**2 + (gaze_y-cy)**2)
            expand = 0.05
            if max(0, bx1-expand) <= gaze_x <= min(1, bx2+expand) and max(0, by1-expand) <= gaze_y <= min(1, by2+expand):
                if dist < best_dist: best_dist, best_pid = dist, pid
            elif dist < self.gaze_to_person_radius and dist < best_dist: best_dist, best_pid = dist, pid
        return (best_pid, "person") if best_pid else (None, "object")
 
@dataclass
class GraphNode:
    person_id: int; observable_state: dict; confidence: dict
 
@dataclass
class GraphEdge:
    source_id: int; target_id: int; edge_type: str; confidence: float; attributes: dict = field(default_factory=dict)
 
@dataclass
class DerivedRelation:
    relation_type: str; participants: List[int]; value: bool; confidence: float; reasoning: str
 
class SocialMindGraph:
    def __init__(self, fov_angle_deg=150):
        self.fov_angle = math.radians(fov_angle_deg); self.nodes = {}; self.edges = []
        self.derived_relations = []; self.visual_access_matrix = {}; self.gazes_at = {}; self.gazed_by = defaultdict(list)
 
    def build(self, active_persons, img_width, img_height):
        self._clear()
        if len(active_persons) < 2:
            for pid, p in active_persons.items(): self._build_node(p)
            return
        for pid, p in active_persons.items(): self._build_node(p)
        self._build_observable_edges(active_persons, img_width, img_height)
        self._compute_visual_access(active_persons)
        self._derive_relations(active_persons)
 
    def _clear(self):
        self.nodes.clear(); self.edges.clear(); self.derived_relations.clear()
        self.visual_access_matrix.clear(); self.gazes_at.clear(); self.gazed_by.clear()
 
    def _build_node(self, person):
        self.nodes[person.person_id] = GraphNode(person.person_id, {
            "position": {"bbox_norm": person.bbox_norm, "center": ((person.bbox_norm[0]+person.bbox_norm[2])/2, (person.bbox_norm[1]+person.bbox_norm[3])/2)},
            "gaze": {"target_person": person.gaze_target_person, "target_position": person.gaze_target_pos, "target_type": person.gaze_target_type, "inout_score": person.inout_score},
            "head_direction": person.head_direction}, {"detection": person.confidence, "gaze": person.gaze_confidence})
 
    def _build_observable_edges(self, active_persons, img_width, img_height):
        pids = list(active_persons.keys())
        for pid in pids:
            person = active_persons[pid]
            if person.gaze_target_person is not None:
                self.edges.append(GraphEdge(pid, person.gaze_target_person, "GAZES_AT", person.gaze_confidence, {"gaze_point": person.gaze_target_pos, "inout_score": person.inout_score}))
                self.gazes_at[pid] = person.gaze_target_person; self.gazed_by[person.gaze_target_person].append(pid)
            else: self.gazes_at[pid] = None
        for i in range(len(pids)):
            for j in range(i+1, len(pids)):
                pi, pj = active_persons[pids[i]], active_persons[pids[j]]
                ci, cj = ((pi.bbox_norm[0]+pi.bbox_norm[2])/2, (pi.bbox_norm[1]+pi.bbox_norm[3])/2), ((pj.bbox_norm[0]+pj.bbox_norm[2])/2, (pj.bbox_norm[1]+pj.bbox_norm[3])/2)
                dist = math.sqrt((ci[0]-cj[0])**2 + (ci[1]-cj[1])**2)
                level = "intimate" if dist<0.15 else "personal" if dist<0.30 else "social" if dist<0.55 else "public"
                self.edges.append(GraphEdge(pids[i], pids[j], "PROXIMATE", min(pi.confidence, pj.confidence), {"distance_norm": dist, "level": level}))
 
    def _compute_visual_access(self, active_persons):
        pids = list(active_persons.keys())
        for pid_i in pids:
            pi, ci = active_persons[pid_i], ((active_persons[pid_i].bbox_norm[0]+active_persons[pid_i].bbox_norm[2])/2, (active_persons[pid_i].bbox_norm[1]+active_persons[pid_i].bbox_norm[3])/2)
            for pid_j in pids:
                if pid_i == pid_j: continue
                pj, cj = active_persons[pid_j], ((active_persons[pid_j].bbox_norm[0]+active_persons[pid_j].bbox_norm[2])/2, (active_persons[pid_j].bbox_norm[1]+active_persons[pid_j].bbox_norm[3])/2)
                can_observe, conf, reason = self._check_visual_access(pi, ci, pj, cj)
                self.visual_access_matrix[(pid_i, pid_j)] = {"can_observe": can_observe, "confidence": conf, "reasoning": reason}
 
    def _check_visual_access(self, person_i, center_i, person_j, center_j):
        if person_i.head_direction is None: return True, 0.3, "Unknown"
        hd_x, hd_y = person_i.head_direction
        dir_x, dir_y = center_j[0]-center_i[0], center_j[1]-center_i[1]
        dir_norm = math.sqrt(dir_x**2 + dir_y**2) + 1e-8
        cos_angle = max(-1, min(1, (hd_x*dir_x/dir_norm) + (hd_y*dir_y/dir_norm)))
        angle, half_fov = math.acos(cos_angle), self.fov_angle / 2
        if angle <= half_fov*0.5: return True, 0.9, f"In central FOV ({math.degrees(angle):.0f}°)"
        elif angle <= half_fov: return True, 0.6, f"In peripheral FOV ({math.degrees(angle):.0f}°)"
        else: return False, 0.8, f"Outside FOV ({math.degrees(angle):.0f}°)"
 
    def _derive_relations(self, active_persons):
        pids = list(active_persons.keys())
        for i in range(len(pids)):
            for j in range(len(pids)):
                if i==j: continue
                va = self.visual_access_matrix.get((pids[i], pids[j]))
                if va: self.derived_relations.append(DerivedRelation("CAN_OBSERVE", [pids[i], pids[j]], va["can_observe"], va["confidence"], va["reasoning"]))
        for i in range(len(pids)):
            for j in range(i+1, len(pids)):
                if self.gazes_at.get(pids[i])==pids[j] and self.gazes_at.get(pids[j])==pids[i]:
                    self.derived_relations.append(DerivedRelation("MUTUAL_GAZE", [pids[i], pids[j]], True, min(active_persons[pids[i]].gaze_confidence, active_persons[pids[j]].gaze_confidence), "Mutual gaze"))
        for pid_i in pids:
            target = self.gazes_at.get(pid_i)
            if target is not None and self.gazes_at.get(target) != pid_i:
                self.derived_relations.append(DerivedRelation("UNRECIPROCATED_ENGAGEMENT", [pid_i, target], True, active_persons[pid_i].gaze_confidence, "Unreciprocated"))
        for pid in pids:
            watchers = self.gazed_by.get(pid, [])
            if len(watchers) >= 2: self.derived_relations.append(DerivedRelation("ATTENTION_CENTER", [pid]+watchers, True, 0.7, "Attended by many"))
            if len(watchers)==0 and (self.gazes_at.get(pid) is None or self.gazes_at.get(self.gazes_at.get(pid))!=pid) and len(pids)>2:
                self.derived_relations.append(DerivedRelation("POSSIBLY_EXCLUDED", [pid], True, 0.5, "Possibly excluded"))
 
    def to_natural_language(self) -> str:
        if len(self.nodes) == 0: return "No persons detected."
        lines = [f"=== Social Mind Graph ({len(self.nodes)} persons) ==="]
        for pid in sorted(self.gazes_at.keys()):
            target = self.gazes_at[pid]
            lines.append(f"P{pid} gazes at {'P'+str(target) if target else self.nodes[pid].observable_state['gaze']['target_type']}")
        for pid_i in sorted(self.nodes.keys()):
            can_see = [f"P{pid_j}" for pid_j in self.nodes if pid_j!=pid_i and self.visual_access_matrix.get((pid_i,pid_j),{}).get("can_observe")]
            if can_see: lines.append(f"P{pid_i} CAN observe: {', '.join(can_see)}")
        for r in self.derived_relations:
            if r.relation_type in ["MUTUAL_GAZE", "UNRECIPROCATED_ENGAGEMENT"]:
                lines.append(f"{r.relation_type}: {', '.join([f'P{p}' for p in r.participants])}")
        return "\n".join(lines)
 
# ---- MODULE C ----
@dataclass
class MentalStateEntry:
    entry_id: str; order: int; entry_type: str; subject: int; predicate: str
    about: Optional[int] = None; value: bool = True; confidence: float = 1.0
    rule_applied: str = ""; premises: List[str] = field(default_factory=list); natural_language: str = ""
 
class MentalStateKnowledgeBase:
    def __init__(self): self.entries, self.by_order, self._counter = {}, defaultdict(list), 0
    def add(self, entry):
        self._counter += 1; entry.entry_id = f"MS_{entry.order}_{self._counter}"
        self.entries[entry.entry_id] = entry; self.by_order[entry.order].append(entry.entry_id)
        return entry.entry_id
    def get_by_order(self, order): return [self.entries[eid] for eid in self.by_order.get(order, [])]
    def to_natural_language(self, min_confidence=0.3):
        lines = []
        for order in sorted(self.by_order.keys()):
            entries = [self.entries[eid] for eid in self.by_order[order] if self.entries[eid].confidence >= min_confidence]
            for e in entries: lines.append(f"Order {e.order} [{e.entry_type}]: {e.natural_language}")
        return "\n".join(lines)
 
class SymbolicToMEngine:
    def __init__(self, max_order=3, confidence_decay=0.85): self.max_order = max_order; self.confidence_decay = confidence_decay
    def reason(self, graph):
        self.mskb = MentalStateKnowledgeBase(); self.graph = graph
        if len(graph.nodes) < 1: return self.mskb
        self._reason_order_0()
        if len(graph.nodes) < 2: return self.mskb
        self._reason_order_1(); self._reason_order_2()
        for order in range(3, self.max_order + 1):
            if self._reason_higher_order(order) == 0: break
        self._infer_social_dynamics(); return self.mskb
    def _reason_order_0(self):
        for pid, node in self.graph.nodes.items():
            gaze_info = node.observable_state.get("gaze", {})
            if gaze_info.get("target_person") is not None:
                self.mskb.add(MentalStateEntry("", 0, "OBSERVABLE", pid, f"gazes_at_P{gaze_info['target_person']}", gaze_info['target_person'], True, node.confidence.get("gaze", 0.5), "Direct", natural_language=f"P{pid} is looking at P{gaze_info['target_person']}"))
            elif gaze_info.get("target_type") in ["object", "out_of_frame"]:
                self.mskb.add(MentalStateEntry("", 0, "OBSERVABLE", pid, f"gazes_{gaze_info['target_type']}", value=True, confidence=node.confidence.get("gaze", 0.5), rule_applied="Direct", natural_language=f"P{pid} is looking {gaze_info['target_type']}"))
    def _reason_order_1(self):
        pids = list(self.graph.nodes.keys())
        for pid_i in pids:
            for pid_j in pids:
                if pid_i == pid_j: continue
                va = self.graph.visual_access_matrix.get((pid_i, pid_j))
                if va is None: continue
                j_gaze_target = self.graph.gazes_at.get(pid_j)
                if va["can_observe"] and j_gaze_target is not None:
                    self.mskb.add(MentalStateEntry("", 1, "BELIEF", pid_i, f"aware_P{pid_j}_looks_P{j_gaze_target}", pid_j, True, va["confidence"]*0.8, "Rule 1.1", natural_language=f"P{pid_i} knows P{pid_j} looks at P{j_gaze_target}"))
                if not va["can_observe"] and j_gaze_target is not None:
                    self.mskb.add(MentalStateEntry("", 1, "IGNORANCE", pid_i, f"unaware_P{pid_j}_looks_P{j_gaze_target}", pid_j, True, va["confidence"], "Rule 1.2", natural_language=f"P{pid_i} does NOT know P{pid_j} looks at P{j_gaze_target}"))
                if not va["can_observe"] and j_gaze_target == pid_i:
                    self.mskb.add(MentalStateEntry("", 1, "IGNORANCE", pid_i, f"unaware_watched_by_P{pid_j}", pid_j, True, va["confidence"], "Rule 1.2b", natural_language=f"P{pid_i} does NOT know P{pid_j} is watching them"))
    def _reason_order_2(self):
        pids = list(self.graph.nodes.keys())
        for entry in self.mskb.get_by_order(1):
            if entry.entry_type == "IGNORANCE" and entry.about is not None:
                for pid_k in pids:
                    if pid_k == entry.subject: continue
                    va_k_sub = self.graph.visual_access_matrix.get((pid_k, entry.subject))
                    va_k_ab = self.graph.visual_access_matrix.get((pid_k, entry.about))
                    va_sub_ab = self.graph.visual_access_matrix.get((entry.subject, entry.about))
                    if va_k_sub and va_k_sub["can_observe"] and va_k_ab and va_k_ab["can_observe"] and va_sub_ab and not va_sub_ab["can_observe"]:
                        self.mskb.add(MentalStateEntry("", 2, "RECURSIVE_BELIEF", pid_k, f"knows_P{entry.subject}_{entry.predicate}", entry.subject, True, entry.confidence*self.confidence_decay, "Rule 2.1", [entry.entry_id], f"P{pid_k} knows P{entry.subject} is unaware"))
        for pid_i in pids:
            target = self.graph.gazes_at.get(pid_i)
            if target is None: continue
            va_target_i = self.graph.visual_access_matrix.get((target, pid_i))
            if va_target_i and not va_target_i["can_observe"]:
                self.mskb.add(MentalStateEntry("", 2, "RECURSIVE_BELIEF", pid_i, f"knows_P{target}_unaware_watched", target, True, va_target_i["confidence"]*0.8, "Rule 2.4", natural_language=f"P{pid_i} knows P{target} doesn't know they are being watched"))
    def _reason_higher_order(self, order):
        pids = list(self.graph.nodes.keys()); new_count = 0
        for entry in self.mskb.get_by_order(order - 1):
            if entry.entry_type != "RECURSIVE_BELIEF" or entry.confidence < 0.3: continue
            for pid_k in pids:
                if pid_k == entry.subject: continue
                va = self.graph.visual_access_matrix.get((pid_k, entry.subject))
                if va and va["can_observe"]:
                    conf = entry.confidence * self.confidence_decay * va["confidence"]
                    if conf > 0.2: 
                        self.mskb.add(MentalStateEntry("", order, "RECURSIVE_BELIEF", pid_k, f"knows_P{entry.subject}_{entry.predicate}", entry.subject, True, conf, f"Rule {order}.1", [entry.entry_id], natural_language=f"P{pid_k} knows P{entry.subject} {entry.predicate.replace('_', ' ')}"))
                        new_count += 1
        return new_count
    def _infer_social_dynamics(self):
        for entry in self.mskb.get_by_order(2):
            if "unaware" in entry.predicate and entry.value:
                self.mskb.add(MentalStateEntry("", 0, "SOCIAL_DYNAMIC", entry.subject, "has_info_advantage", entry.about, True, entry.confidence, "Dynamic", natural_language=f"P{entry.subject} has info advantage over P{entry.about}"))
 
# ---- MODULE D ----
class LLMReasoningModule:
    def __init__(self, model_name=MODEL_NAME, max_retries=3): self.model_name = model_name; self.max_retries = max_retries
 
    def _encode_image(self, pil_image, max_size=512):
        w, h = pil_image.size
        if max(w, h) > max_size: pil_image = pil_image.resize((int(w*max_size/max(w,h)), int(h*max_size/max(w,h))), Image.LANCZOS)
        buffer = BytesIO(); pil_image.save(buffer, format="JPEG", quality=85); return base64.b64encode(buffer.getvalue()).decode("utf-8")
 
    def _call_llm(self, messages, temperature=0.3, max_tokens=2000):
        for attempt in range(self.max_retries):
            try: return client.chat.completions.create(model=self.model_name, messages=messages, temperature=temperature, max_tokens=max_tokens).choices[0].message.content
            except Exception as e:
                if attempt < self.max_retries - 1: time.sleep(2 ** attempt)
                else: return f"[LLM Error: {str(e)}]"
 
    def _call_llm_with_image(self, sys_prompt, user_prompt, pil_image, temp=0.3, max_tokens=1500):
        img_b64 = self._encode_image(pil_image)
        messages = [{"role": "system", "content": sys_prompt}, {"role": "user", "content": [{"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}}, {"type": "text", "text": user_prompt}]}]
        return self._call_llm(messages, temp, max_tokens)
 
    def complementary_reasoning(self, mskb, graph, pil_image):
        sys_prompt = "You are a social cognition expert. Based on the verified mental states and image, briefly infer: 1. Scene context. 2. Specific intentions. Do not contradict verified facts. Label inferences as [INFERRED]."
        user_prompt = f"=== VERIFIED MENTAL STATES ===\n{mskb.to_natural_language()}\n=== GRAPH ===\n{graph.to_natural_language()}\nBriefly provide context and intent."
        return self._call_llm_with_image(sys_prompt, user_prompt, pil_image)
 

######################################################################
######################################################################
    def answer_social_iq(self, question, answer_choices, mskb, graph, pil_image, comp_analysis=""):
        sys_prompt = """You are an expert social scene analyst. Select the best answer.
RESPONSE FORMAT (JSON):
{"selected_answer_index": 0, "reasoning": "step-by-step reasoning", "tom_order_required": 0}
RULES: Ground reasoning in VERIFIED mental states. Consider Visual Access for perspective-taking.

"""
        choices_text = "\n".join([f"[{c.get('index', i)}] {c['text']}" for i, c in enumerate(answer_choices)])
        user_prompt = f"=== GRAPH ===\n{graph.to_natural_language()}\n=== VERIFIED MENTAL STATES ===\n{mskb.to_natural_language()}\n=== CONTEXT ===\n{comp_analysis if comp_analysis else 'N/A'}\n=== QUESTION ===\n{question}\n=== CHOICES ===\n{choices_text}"
        raw = self._call_llm_with_image(sys_prompt, user_prompt, pil_image, temp=0.1)
        try:
            json_match = re.search(r'\{[\s\S]*\}', raw)
            if json_match: return json.loads(json_match.group())
        except: pass
        return {"selected_answer_index": -1, "reasoning": raw, "tom_order_required": -1}
######################################################################
######################################################################
 
# ---- Parser ----
class SocialIQParser:
    @staticmethod
    def parse_qa_file_grouped(filepath: str) -> Dict[str, List[dict]]:
        grouped = defaultdict(list)
        with open(filepath, 'r', encoding='utf-8') as f:
            content = f.read().strip()
            data_list = json.loads(content) if content.startswith('[') else [json.loads(line) for line in content.split('\n') if line.strip()]
        for data in data_list:
            vid_name = data.get("vid_name")
            correct_idx = data.get("answer_idx", -1)
            current_q = {"question_id": data.get("qid"), "question": data.get("q"), "answers": []}
            for i in range(4):
                if f"a{i}" in data:
                    current_q["answers"].append({"index": i, "text": data[f"a{i}"], "is_correct": (i == correct_idx)})
            grouped[vid_name].append(current_q)
        return grouped
 

# ---- Pipeline (多帧版本) ----
class SocialMindPipeline:
    def __init__(self, gazelle_model, gazelle_transform, device, face_detector,
                 max_persons=5, inout_thresh=0.5, yolo_conf=0.7, max_tom_order=3,
                 num_keyframes=5):
        self.gaze_detector = GazeDetector(gazelle_model, gazelle_transform, device, inout_thresh)
        self.yolo = face_detector
        self.yolo_conf = yolo_conf
        self.max_persons = max_persons
        self.graph = SocialMindGraph()
        self.tom_engine = SymbolicToMEngine(max_tom_order)
        self.llm_module = LLMReasoningModule()
        self.tracker = PersonTracker(max_persons=max_persons)
        self.num_keyframes = num_keyframes  # 选几帧

        # 实时写入
        self.output_json_path = None
        self.all_results = []
        self.total_correct = 0
        self.total_questions = 0

    def _update_results_json(self):
        if not self.output_json_path:
            return
        final_output = {
            "dataset_accuracy": (
                f"{self.total_correct}/{self.total_questions} "
                f"({self.total_correct / self.total_questions * 100:.1f}%)"
                if self.total_questions > 0 else "0/0 (0.0%)"
            ),
            "results": self.all_results
        }
        with open(self.output_json_path, 'w', encoding='utf-8') as f:
            json.dump(final_output, f, indent=2, ensure_ascii=False)

    def process_dataset(self, video_dir: str, qa_file_path: str, output_json_path: str):
        print("Loading QA data...")
        grouped_qa = SocialIQParser.parse_qa_file_grouped(qa_file_path)
        video_files = [f for f in os.listdir(video_dir) if f.endswith(('.mp4', '.avi', '.mov'))]

        self.output_json_path = output_json_path
        self.all_results = []
        self.total_correct = 0
        self.total_questions = 0
        self._update_results_json()

        for idx, vid_file in enumerate(video_files):
            vid_name = os.path.splitext(vid_file)[0]
            if vid_name not in grouped_qa:
                continue
            video_path = os.path.join(video_dir, vid_file)
            qa_data = grouped_qa[vid_name]
            print(f"\n[{idx + 1}/{len(video_files)}] Processing {vid_name} ({len(qa_data)} Qs)...")

            self.tracker.reset()

            # 1. 提取多帧关键帧
            multi_kf = self._extract_multi_keyframes(video_path)
            if "error" in multi_kf:
                self.all_results.append({"vid_name": vid_name, "error": multi_kf["error"], "qa_results": []})
                self._update_results_json()
                continue

            primary = multi_kf["primary"]
            aggregated_context = multi_kf["aggregated_context"]

            # 2. D1: 补充推理（用primary帧的图片+聚合context）
            comp = self.llm_module.complementary_reasoning(
                primary["mskb"], primary["graph"], primary["pil_image"]
            )
            # 拼接多帧聚合信息到补充分析中
            full_context = comp + "\n\n" + aggregated_context

            vid_result_entry = {"vid_name": vid_name, "accuracy": "0/0 (0.0%)",
                                "num_keyframes_used": multi_kf["num_keyframes"],
                                "qa_results": []}
            self.all_results.append(vid_result_entry)

            # 3. 逐题推理
            vid_correct = 0
            for q_idx, q in enumerate(qa_data):
                res = self.llm_module.answer_social_iq(
                    q["question"], q["answers"],
                    primary["mskb"], primary["graph"],
                    primary["pil_image"], full_context
                )
                pred_idx = res.get("selected_answer_index", -1)
                gt_idx = next((ans["index"] for ans in q["answers"] if ans.get("is_correct")), -1)
                is_correct = (pred_idx == gt_idx)

                qa_result = {
                    "question_id": q["question_id"],
                    "question": q["question"],
                    "predicted_index": pred_idx,
                    "ground_truth_index": gt_idx,
                    "is_correct": is_correct,
                    "reasoning": res.get("reasoning", ""),
                    "tom_order_required": res.get("tom_order_required", -1)
                }
                vid_result_entry["qa_results"].append(qa_result)
                self.total_questions += 1
                if is_correct:
                    self.total_correct += 1
                    vid_correct += 1

                mark = "✓" if is_correct else "✗"
                print(f"  [{q_idx + 1}/{len(qa_data)}] {mark} Pred: {pred_idx}, GT: {gt_idx} "
                      f"| ToM: {res.get('tom_order_required', -1)} "
                      f"| Reason: {res.get('reasoning', '')[:80]}...")

                current_q_count = len(vid_result_entry["qa_results"])
                vid_result_entry["accuracy"] = (
                    f"{vid_correct}/{current_q_count} "
                    f"({vid_correct / current_q_count * 100:.1f}%)"
                )
                self._update_results_json()

        print(f"\nDataset evaluation complete! "
              f"Overall Acc: {self.total_correct}/{self.total_questions} "
              f"({self.total_correct / self.total_questions * 100:.1f}%)"
              if self.total_questions > 0 else "No questions processed.")

    def _extract_multi_keyframes(self, video_path: str) -> dict:
        """
        处理视频，提取多个关键帧的Graph和MSKB，聚合成统一context

        返回:
        {
            "primary": {"pil_image", "graph", "mskb"},  # 最佳帧（用于图片输入）
            "aggregated_context": str,                    # 多帧聚合的文本描述
            "num_keyframes": int
        }
        """
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            return {"error": "Cannot open video"}

        fps = int(cap.get(cv2.CAP_PROP_FPS)) or 30
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        # 每秒抽1帧
        interval = max(1, fps)
        all_candidates = []
        frame_count = 0

        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame_count += 1
            if frame_count % interval != 0:
                continue

            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_image = Image.fromarray(rgb_frame)

            results = self.yolo(rgb_frame, verbose=False, conf=self.yolo_conf)
            boxes, confs = [], []
            if len(results) > 0 and len(results[0].boxes) > 0:
                boxes_np = results[0].boxes.xyxy.cpu().numpy()
                confs_np = results[0].boxes.conf.cpu().numpy()
                areas = (boxes_np[:, 2] - boxes_np[:, 0]) * (boxes_np[:, 3] - boxes_np[:, 1])
                for sort_idx in np.argsort(areas)[::-1][:self.max_persons]:
                    boxes.append(boxes_np[sort_idx].tolist())
                    confs.append(float(confs_np[sort_idx]))

            active_persons = self.tracker.update(boxes, confs, frame_count, width, height)

            if len(active_persons) > 0:
                active_persons = self.gaze_detector.detect(pil_image, active_persons)

            self.graph.build(active_persons, width, height)

            # 放宽条件：≥1人也保留（但评分更低）
            if len(active_persons) >= 1:
                mskb = self.tom_engine.reason(self.graph)
                num_interactions = len([e for e in self.graph.edges if e.edge_type == "GAZES_AT"])
                timestamp = frame_count / fps if fps > 0 else 0

                # 生成人物位置描述
                person_desc = self._describe_persons(active_persons)

                all_candidates.append({
                    "frame_idx": frame_count,
                    "timestamp": timestamp,
                    "pil_image": pil_image.copy(),
                    "graph_nl": self.graph.to_natural_language(),
                    "mskb_nl": mskb.to_natural_language(),
                    "person_desc": person_desc,
                    "num_persons": len(active_persons),
                    "num_interactions": num_interactions,
                    # 用于primary帧重建
                    "_boxes": boxes,
                    "_confs": confs,
                    "_width": width,
                    "_height": height,
                })

        cap.release()

        if not all_candidates:
            return {"error": "No valid keyframes with detected persons"}

        # 从候选帧中选择多帧（覆盖不同时段）
        selected = self._select_diverse_keyframes(all_candidates, self.num_keyframes)

        # 选primary帧（评分最高的，用于图片输入和重建graph/mskb）
        primary_kf = max(selected, key=lambda x: x["num_persons"] * 2 + x["num_interactions"] * 3)

        # 重建primary帧的graph和mskb
        primary_rebuilt = self._rebuild_frame(primary_kf)

        # 聚合所有选中帧的信息
        aggregated = self._aggregate_keyframes(selected, primary_kf["frame_idx"])

        return {
            "primary": primary_rebuilt,
            "aggregated_context": aggregated,
            "num_keyframes": len(selected)
        }

    def _select_diverse_keyframes(self, candidates: List[dict], top_k: int) -> List[dict]:
        """
        从候选帧中选择top_k帧，覆盖视频不同时段

        策略：将视频分成top_k个时间段，每段选评分最高的帧
        """
        if len(candidates) <= top_k:
            return candidates

        # 按帧号排序
        sorted_cands = sorted(candidates, key=lambda x: x["frame_idx"])

        # 分段
        segment_size = max(1, len(sorted_cands) // top_k)
        selected = []

        for i in range(top_k):
            start = i * segment_size
            end = start + segment_size if i < top_k - 1 else len(sorted_cands)
            segment = sorted_cands[start:end]

            if segment:
                # 每段选评分最高的（优先≥2人的帧）
                best = max(segment, key=lambda x: x["num_persons"] * 2 + x["num_interactions"] * 3)
                selected.append(best)

        return selected

    def _describe_persons(self, active_persons: Dict[int, 'TrackedPerson']) -> str:
        """为每个人生成位置和外观描述，帮助LLM匹配ID"""
        if not active_persons:
            return ""

        lines = []
        for pid, person in sorted(active_persons.items()):
            cx = (person.bbox_norm[0] + person.bbox_norm[2]) / 2
            cy = (person.bbox_norm[1] + person.bbox_norm[3]) / 2
            bbox_h = person.bbox_norm[3] - person.bbox_norm[1]
            bbox_w = person.bbox_norm[2] - person.bbox_norm[0]

            # 水平位置
            if cx < 0.33:
                h_pos = "on the left"
            elif cx < 0.66:
                h_pos = "in the center"
            else:
                h_pos = "on the right"

            # 大小/远近
            if bbox_h > 0.4:
                size = "close-up"
            elif bbox_h > 0.2:
                size = "medium shot"
            elif bbox_h > 0.1:
                size = "medium-far"
            else:
                size = "far/small"

            # 注视
            if person.gaze_target_person is not None:
                gaze = f"looking at P{person.gaze_target_person}"
            elif person.gaze_target_type == "object":
                gaze = "looking at an object"
            elif person.gaze_target_type == "out_of_frame":
                gaze = "looking out of frame"
            else:
                gaze = "gaze unknown"

            lines.append(f"  P{pid}: {h_pos}, {size}, {gaze}")

        return "Person descriptions:\n" + "\n".join(lines)

    def _rebuild_frame(self, kf: dict) -> dict:
        """对选中的关键帧重新跑检测→gaze→graph→ToM，得到新鲜的对象"""
        pil_image = kf["pil_image"]
        rgb_frame = np.array(pil_image)
        w, h = kf["_width"], kf["_height"]

        # 重新检测
        results = self.yolo(rgb_frame, verbose=False, conf=self.yolo_conf)
        boxes, confs = [], []
        if len(results) > 0 and len(results[0].boxes) > 0:
            boxes_np = results[0].boxes.xyxy.cpu().numpy()
            confs_np = results[0].boxes.conf.cpu().numpy()
            areas = (boxes_np[:, 2] - boxes_np[:, 0]) * (boxes_np[:, 3] - boxes_np[:, 1])
            for sort_idx in np.argsort(areas)[::-1][:self.max_persons]:
                boxes.append(boxes_np[sort_idx].tolist())
                confs.append(float(confs_np[sort_idx]))

        temp_tracker = PersonTracker(max_persons=self.max_persons)
        active_persons = temp_tracker.update(boxes, confs, 1, w, h)

        if len(active_persons) > 0:
            active_persons = self.gaze_detector.detect(pil_image, active_persons)

        fresh_graph = SocialMindGraph()
        fresh_graph.build(active_persons, w, h)

        fresh_mskb = self.tom_engine.reason(fresh_graph)

        return {
            "pil_image": pil_image,
            "graph": fresh_graph,
            "mskb": fresh_mskb,
        }

    def _aggregate_keyframes(self, keyframes: List[dict], primary_frame_idx: int) -> str:
        """
        将多帧的分析结果聚合成一段文本context

        包含：
        1. 每帧的时间戳+人物+注视关系+ToM推理
        2. 跨帧的变化总结
        3. 人物位置描述（帮助LLM对应ID）
        """
        lines = []
        lines.append("=" * 50)
        lines.append(f"MULTI-FRAME ANALYSIS ({len(keyframes)} keyframes)")
        lines.append("=" * 50)

        # 逐帧描述
        for i, kf in enumerate(sorted(keyframes, key=lambda x: x["frame_idx"])):
            is_primary = (kf["frame_idx"] == primary_frame_idx)
            marker = " ★PRIMARY" if is_primary else ""
            lines.append(f"\n--- Frame #{kf['frame_idx']} ({kf['timestamp']:.1f}s){marker} ---")
            lines.append(f"Persons detected: {kf['num_persons']}, "
                         f"Gaze interactions: {kf['num_interactions']}")

            # 人物描述
            if kf.get("person_desc"):
                lines.append(kf["person_desc"])

            # 注视关系
            lines.append(f"Social Graph:\n{kf['graph_nl']}")

            # ToM推理（只显示关键部分，避免太长）
            mskb_text = kf["mskb_nl"]
            if mskb_text:
                # 截取前500字符，避免token浪费
                if len(mskb_text) > 500:
                    mskb_text = mskb_text[:500] + "..."
                lines.append(f"Mental States:\n{mskb_text}")

        # 跨帧变化总结
        lines.append("\n" + "=" * 50)
        lines.append("TEMPORAL SUMMARY")
        lines.append("=" * 50)

        # 统计各帧的人数变化
        person_counts = [kf["num_persons"] for kf in keyframes]
        interaction_counts = [kf["num_interactions"] for kf in keyframes]

        lines.append(f"Person count across frames: {person_counts}")
        lines.append(f"Interaction count across frames: {interaction_counts}")

        if max(person_counts) > min(person_counts):
            lines.append("Note: Number of visible persons changes across time "
                         "(camera angle changes or people entering/leaving)")

        # 收集所有帧中出现的注视模式
        all_gaze_patterns = set()
        for kf in keyframes:
            graph_text = kf.get("graph_nl", "")
            for line in graph_text.split("\n"):
                if "gazes at" in line.lower() or "GAZES_AT" in line:
                    all_gaze_patterns.add(line.strip())

        if all_gaze_patterns:
            lines.append(f"\nAll observed gaze patterns across time:")
            for pattern in sorted(all_gaze_patterns):
                lines.append(f"  {pattern}")

        return "\n".join(lines)


if __name__ == "__main__":
    VIDEO_DIR = "/Users/zehao/Desktop/HAI/MultiParty/Social-IQ-2/siq2/video"
    QA_FILE = "/Users/zehao/Desktop/HAI/MultiParty/Social-IQ-2/siq2/qa/all.json"
    OUTPUT_JSON = "/Users/zehao/Desktop/HAI/MultiParty/code/results/m4改prompt_results.json"

    pipeline = SocialMindPipeline(
        gazelle_model=gazelle_model, gazelle_transform=gazelle_transform,
        device=device, face_detector=face_detector,
        max_persons=5, inout_thresh=0.5, yolo_conf=0.7, max_tom_order=3,
        num_keyframes=20  # 每个视频选10帧
    )
    pipeline.process_dataset(
        video_dir=VIDEO_DIR, qa_file_path=QA_FILE, output_json_path=OUTPUT_JSON
    )


Using device: cpu


Using cache found in /Users/zehao/Desktop/HAI/MultiParty/code/gazelle-main/fkryan_gazelle_main
Using cache found in /Users/zehao/Desktop/HAI/MultiParty/code/gazelle-main/facebookresearch_dinov2_main


Loading QA data...

[2/1242] Processing ZP8ACbJ677I (5 Qs)...
  [1/5] ✗ Pred: 0, GT: 3 | ToM: 0 | Reason: The image displays a 'Before and After' comparison for a Lancôme mascara set on ...
  [2/5] ✓ Pred: 3, GT: 3 | ToM: 0 | Reason: Step 1 - CHECK VERIFIED FACTS: The context explicitly states the scene is a 'tel...
  [3/5] ✓ Pred: 3, GT: 3 | ToM: 1 | Reason: Step 1 - CHECK VERIFIED FACTS: The provided Social Mind Graph and Verified Menta...
  [4/5] ✓ Pred: 1, GT: 1 | ToM: 0 | Reason: Step 1 - CHECK VERIFIED FACTS: No specific verified mental states regarding the ...
  [5/5] ✗ Pred: 3, GT: 2 | ToM: 0 | Reason: The image displays a split-screen 'Before' and 'After' comparison typical of a H...

[3/1242] Processing SWNXZbasPvQ (11 Qs)...
  [1/11] ✓ Pred: 0, GT: 0 | ToM: 1 | Reason: Step 1 - CHECK VERIFIED FACTS: The verified mental states and context indicate t...
  [2/11] ✓ Pred: 0, GT: 0 | ToM: 1 | Reason: Step 1 - CHECK VERIFIED FACTS: The verified mental states indicate that P1 (man 

KeyboardInterrupt: 

In [14]:
# ================================================================
#  评估模块 + VLM Baseline 对比
#  
#  功能：
#  1. 解析Social-IQ的ground truth
#  2. 评估pipeline的QA准确率
#  3. 用VLM直接推理作为baseline对比
#  4. 生成对比报告
# ================================================================

import os
import json
import time
import base64
import re
from io import BytesIO
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass
from PIL import Image
import numpy as np
import cv2

import openai

# ================= API 配置 =================
API_KEY = "sk-bbf9f24ff4194920a43e15749a2dad29"           # ← 填写你的API Key
API_BASE = "https://dashscope.aliyuncs.com/compatible-mode/v1"  # ← 填写你的API Base URL
MODEL_NAME = "qwen-vl-plus-latest"
# 视觉模型是Qwen-VL-3.6，语言模型是qwen-plus-latest

client = openai.OpenAI(api_key=API_KEY, base_url=API_BASE)

class VideoSampler:
    """
    统一的视频帧采样器
    确保Pipeline和Baseline看到完全相同的帧
    """
    
    def __init__(self, video_path: str):
        self.video_path = video_path
        
        cap = cv2.VideoCapture(video_path)
        self.fps = int(cap.get(cv2.CAP_PROP_FPS))
        self.width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        self.height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        self.total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        self.duration = self.total_frames / self.fps if self.fps > 0 else 0
        cap.release()
        
        print(f"Video: {self.width}x{self.height}, {self.fps}FPS, "
              f"{self.total_frames} frames, {self.duration:.1f}s")
    
    def sample_frames(self, num_frames: int = 8, 
                       strategy: str = "uniform") -> List[dict]:
        """
        采样指定数量的帧
        """
        if strategy == "uniform":
            # 均匀分布，排除首尾各5%
            start = int(self.total_frames * 0.05)
            end = int(self.total_frames * 0.95)
            indices = np.linspace(start, end, num_frames, dtype=int).tolist()
        elif strategy == "keypoints":
            positions =[0.15, 0.25, 0.35, 0.45, 0.55, 0.65, 0.75, 0.85]
            indices =[int(self.total_frames * p) for p in positions[:num_frames]]
        else:
            raise ValueError(f"Unknown strategy: {strategy}")
        
        # 去重并排序
        indices = sorted(set(indices))
        return self._extract_frames(indices)
    
    def _extract_frames(self, indices: List[int]) -> List[dict]:
        """提取指定帧"""
        cap = cv2.VideoCapture(self.video_path)
        frames =[]
        
        for idx in indices:
            idx = max(0, min(idx, self.total_frames - 1))
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            
            if ret:
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                pil = Image.fromarray(rgb)
                frames.append({
                    "frame_idx": idx,
                    "pil_image": pil,
                    "timestamp": idx / self.fps if self.fps > 0 else 0
                })
        
        cap.release()
        print(f"Sampled {len(frames)} frames at indices: {[f['frame_idx'] for f in frames]}")
        return frames


# ================================================================
#  多帧图像处理工具
# ================================================================

class MultiFrameProcessor:
    """处理VLM的多图输入"""
    
    @staticmethod
    def encode_image(pil_image: Image.Image, 
                      max_size: int = 768) -> str:
        """编码为base64"""
        w, h = pil_image.size
        if max(w, h) > max_size:
            ratio = max_size / max(w, h)
            pil_image = pil_image.resize(
                (int(w * ratio), int(h * ratio)), Image.LANCZOS
            )
        buf = BytesIO()
        pil_image.save(buf, format="JPEG", quality=85)
        return base64.b64encode(buf.getvalue()).decode("utf-8")


# ================================================================
#  公平的VLM Baseline（多帧版本）
# ================================================================

class FairVLMBaseline:
    """
    极简 VLM Baseline
    输入：多帧分别作为多图独立传入
    推理：直接推理 (Direct)
    """
    
    def __init__(self, model_name: str = MODEL_NAME, 
                 max_retries: int = 3):
        self.model_name = model_name
        self.max_retries = max_retries
        self.processor = MultiFrameProcessor()
    
    def _call_api(self, messages: list, temperature: float = 0.1,
                   max_tokens: int = 2000) -> str:
        for attempt in range(self.max_retries):
            try:
                response = client.chat.completions.create(
                    model=self.model_name,
                    messages=messages,
                    temperature=temperature,
                    max_tokens=max_tokens
                )
                return response.choices[0].message.content
            except Exception as e:
                print(f"    API error (attempt {attempt+1}): {e}")
                if attempt < self.max_retries - 1:
                    time.sleep(2 ** attempt)
                else:
                    return f"[Error: {str(e)}]"
    
    def _parse_response(self, raw: str) -> dict:
        try:
            match = re.search(r'\{[\s\S]*?\}', raw)
            if match:
                result = json.loads(match.group())
                result.setdefault("selected_answer_index", -1)
                return result
        except json.JSONDecodeError:
            pass
        
        idx_match = re.search(r'(?:select|answer|index)[^\d]*(\d+)',
                               raw, re.IGNORECASE)
        if idx_match:
            return {"selected_answer_index": int(idx_match.group(1)),
                    "reasoning": raw}
        
        return {"selected_answer_index": -1, "reasoning": raw, "raw": raw}
    
    def _multi_image_input(self, frames: List[dict],
                            system_prompt: str, user_prompt: str,
                            max_images: int = 8,
                            temperature: float = 0.1) -> str:
        """将多帧作为多张图像输入"""
        if len(frames) > max_images:
            indices = np.linspace(0, len(frames)-1, max_images, dtype=int)
            frames = [frames[i] for i in indices]
        
        image_contents =[]
        for f in frames:
            b64 = self.processor.encode_image(f["pil_image"], max_size=512)
            image_contents.append({
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{b64}"}
            })
        
        frame_desc = ", ".join([
            f"Frame {f['frame_idx']} ({f['timestamp']:.1f}s)" 
            for f in frames
        ])
        
        messages =[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content":[
                *image_contents,
                {"type": "text", "text": 
                 f"The above {len(frames)} images are sampled from a video "
                 f"at: {frame_desc}\n\n{user_prompt}"}
            ]}
        ]
        return self._call_api(messages, temperature, max_tokens=2000)
    
    def _build_prompts(self, question: str, choices: List[dict], num_frames: int) -> Tuple[str, str]:
        """构建 Direct 推理 prompt"""
        choices_text = "\n".join([
            f"  [{c['index']}] {c['text']}" for c in choices
        ])
        
        frame_context = (
            f"You are viewing {num_frames} frames sampled from a video "
            f"of a multi-party social interaction. "
            f"Consider the temporal progression across frames."
        )
        
        system_prompt = f"""You are analyzing a social interaction scene.
{frame_context}
Answer the multiple-choice question by selecting the best answer.

Respond ONLY in this JSON format:
{{"selected_answer_index": <int>, "selected_answer_text": "<text>", "reasoning": "<brief reasoning>"}}"""
        
        user_prompt = f"""Question: {question}

Answer Choices:
{choices_text}

Select the best answer by index."""

        return system_prompt, user_prompt
    
    def run_all_baselines(self, questions: List[dict],
                           frames: List[dict],
                           delay: float = 2.0) -> Dict[str, List[dict]]:
        """
        运行 baseline 矩阵，目前精简为只有一个: multi_direct
        """
        baseline_configs = [
            ("multi_direct", "multi_image", "direct"),
        ]
        
        results = {name:[] for name, _, _ in baseline_configs}
        
        for q_idx, q in enumerate(questions):
            print(f"\n  [{q['question_id']}] {q['question'][:55]}...")
            
            for name, input_mode, reasoning_mode in baseline_configs:
                print(f"    {name}...", end=" ", flush=True)
                
                # 直接获取 prompt 并调用 multi_image_input
                sys_prompt, user_prompt = self._build_prompts(
                    q["question"], q["answers"], len(frames)
                )
                
                raw_ans = self._multi_image_input(frames, sys_prompt, user_prompt)
                ans = self._parse_response(raw_ans)
                ans["method"] = name
                ans["num_frames"] = len(frames)
                
                results[name].append(ans)
                
                sel_idx = ans.get("selected_answer_index", -1)
                if 0 <= sel_idx < len(q["answers"]):
                    # label = q["answers"][sel_idx]["label"]
                    is_correct = q["answers"][sel_idx].get("is_correct", False)

                    mark = "✓" if is_correct == "a" else "✗"
                else:
                    mark = "?"
                print(f"{mark}")
                
                time.sleep(delay)
        
        return results


# ================================================================
#  评估器
# ================================================================

class SocialIQEvaluator:
    
    @staticmethod
    def parse_qa_file(filepath: str, target_vid_name: str = None) -> List[dict]:
        """解析 Social-IQ 2.0 的 JSON/JSONL QA文件"""
        questions = []
        
        with open(filepath, 'r', encoding='utf-8') as f:
            content = f.read().strip()
            
            # 兼容处理：检查是标准JSON列表 还是 JSON Lines
            if content.startswith('['):
                data_list = json.loads(content)
            else:
                data_list = [json.loads(line) for line in content.split('\n') if line.strip()]
        
        for data in data_list:
            # 如果指定了 vid_name，过滤掉不属于该视频的QA
            if target_vid_name and data.get("vid_name") != target_vid_name:
                continue
            
            q_id = data.get("qid")
            q_text = data.get("q")
            correct_idx = data.get("answer_idx", -1)
            
            current_q = {
                "question_id": q_id,
                "question": q_text,
                "vid_name": data.get("vid_name"),
                "ans_corr": data.get("ans_corr", ""),  # 保存正确答案文本供参考
                "answers": [],
                "correct_indices": [],
                "incorrect_indices": []
            }
            
            # Social-IQ 2 默认有4个选项 a0, a1, a2, a3
            for i in range(4):
                ans_key = f"a{i}"
                if ans_key in data:
                    text = data[ans_key]
                    is_correct = (i == correct_idx)
                    label = 'a' if is_correct else 'i'  # 兼容旧代码的 label
                    
                    current_q["answers"].append({
                        "index": i, 
                        "label": label,
                        "text": text, 
                        "is_correct": is_correct
                    })
                    
                    if is_correct:
                        current_q["correct_indices"].append(i)
                    else:
                        current_q["incorrect_indices"].append(i)
            
            questions.append(current_q)
            
        return questions
    
    @staticmethod
    def evaluate_batch(questions: List[dict], 
                       model_answers: List[dict]) -> dict:
        results =[]
        correct = 0
        
        for q, ans in zip(questions, model_answers):
            sel_idx = ans.get("selected_answer_index", -1)
            if sel_idx is None:
                sel_idx = -1
            sel_idx = int(sel_idx)
            
            if 0 <= sel_idx < len(q["answers"]):
                is_correct = q["answers"][sel_idx]["is_correct"]
                sel_label = q["answers"][sel_idx]["label"]
                sel_text = q["answers"][sel_idx]["text"]
            else:
                is_correct = False
                sel_label = "invalid"
                sel_text = "N/A"
            
            results.append({
                "question_id": q["question_id"],
                "correct": is_correct,
                "selected_label": sel_label,
                "selected_text": sel_text
            })
            if is_correct:
                correct += 1
        
        total = len(questions)
        return {
            "accuracy": correct / total if total > 0 else 0,
            "correct": correct,
            "total": total,
            "per_question": results
        }


# ================================================================
#  完整的公平对比实验
# ================================================================

class FairComparisonExperiment:
    """
    公平对比实验
    保证：Pipeline 和 Baseline 看到完全相同的帧
    """
    
    def __init__(self):
        self.evaluator = SocialIQEvaluator()
        self.vlm = FairVLMBaseline(model_name=MODEL_NAME)
    
    def run(self, 
            video_path: str,
            qa_file_path: str,
            pipeline_result_path: str = None,
            output_path: str = None,
            num_sample_frames: int = 8,
            api_delay: float = 2.0) -> dict:
        
        print("=" * 70)
        print("  FAIR COMPARISON EXPERIMENT")
        print("  Pipeline and all baselines see the SAME frames")
        print("=" * 70)
        
        # ===== 1. 解析QA =====
        # questions = self.evaluator.parse_qa_file(qa_file_path)
        # print(f"\nLoaded {len(questions)} questions")

        vid_name = os.path.splitext(os.path.basename(video_path))[0]
        questions = self.evaluator.parse_qa_file(qa_file_path, target_vid_name=vid_name)
        print(f"\nLoaded {len(questions)} questions for video: {vid_name}")
        
        # ===== 2. 统一采样帧 =====
        print(f"\n--- Sampling {num_sample_frames} frames ---")
        sampler = VideoSampler(video_path)
        
        pipeline_results = None
        if pipeline_result_path and os.path.exists(pipeline_result_path):
            with open(pipeline_result_path, 'r') as f:
                pipeline_results = json.load(f)
        
        # 均匀采样（这些帧Pipeline和Baseline都会看到）
        frames = sampler.sample_frames(
            num_frames=num_sample_frames, 
            strategy="uniform"
        )
        print(f"  All baselines will see frames: {[f['frame_idx'] for f in frames]}")
        
        # ===== 3. 收集Pipeline结果 =====
        print("\n--- Pipeline Results ---")
        pipeline_answers =[]
        if pipeline_results and "qa_results" in pipeline_results:
            for qa_r in pipeline_results["qa_results"]:
                pipeline_answers.append(qa_r["model_response"])
            print(f"  Loaded {len(pipeline_answers)} pipeline answers")
        else:
            print("  No pipeline results available")
            pipeline_answers = [{"selected_answer_index": -1}] * len(questions)
        
        # ===== 4. 运行VLM Baseline =====
        print("\n--- Running VLM Baseline ---")
        print(f"  Baseline sees {len(frames)} frames directly")
        
        baseline_results = self.vlm.run_all_baselines(
            questions, frames, delay=api_delay
        )
        
        # ===== 5. 评估所有方法 =====
        print("\n" + "=" * 70)
        print("  EVALUATION")
        print("=" * 70)
        
        all_evals = {}
        
        # Pipeline
        ev_pipeline = self.evaluator.evaluate_batch(questions, pipeline_answers)
        all_evals["Ours (Neuro-Symbolic)"] = ev_pipeline
        
        # Baselines
        for method_name, answers in baseline_results.items():
            ev = self.evaluator.evaluate_batch(questions, answers)
            all_evals[method_name] = ev
        
        # ===== 6. 生成报告 =====
        report = self._generate_report(all_evals, questions, len(frames))
        print(report)
        
        # ===== 7. 保存 =====
        output = {
            "experiment_config": {
                "video": video_path,
                "qa_file": qa_file_path,
                "num_frames_sampled": len(frames),
                "frame_indices": [f["frame_idx"] for f in frames],
                "model": MODEL_NAME,
                "fairness": "All methods see identical frames"
            },
            "evaluations": {
                method: {
                    "accuracy": ev["accuracy"],
                    "correct": ev["correct"],
                    "total": ev["total"],
                    "per_question": ev["per_question"]
                }
                for method, ev in all_evals.items()
            },
            "raw_responses": {
                method:[
                    {"question_id": questions[i]["question_id"],
                     "response": ans}
                    for i, ans in enumerate(answers)
                ]
                for method, answers in baseline_results.items()
            },
            "report": report
        }
        
        if output_path:
            with open(output_path, 'w', encoding='utf-8') as f:
                json.dump(output, f, indent=2, ensure_ascii=False, default=str)
            print(f"\nResults saved to: {output_path}")
        
        return output
    
    def _generate_report(self, all_evals: dict, 
                          questions: list,
                          num_frames: int) -> str:
        """生成对比报告"""
        lines =[]
        lines.append("\n" + "=" * 70)
        lines.append("  FAIR COMPARISON REPORT")
        lines.append(f"  All methods see the same {num_frames} frames")
        lines.append("=" * 70)
        
        # ===== 分组显示 =====
        groups = {
            "Our Method":[],
            "Multi-Image (N帧→N图)":[]
        }
        
        for method, ev in all_evals.items():
            if "Ours" in method or "Neuro" in method:
                groups["Our Method"].append((method, ev))
            elif "multi" in method:
                groups["Multi-Image (N帧→N图)"].append((method, ev))
        
        for group_name, methods in groups.items():
            if not methods:
                continue
            
            lines.append(f"\n  ┌─ {group_name} {'─' * (50 - len(group_name))}┐")
            for method, ev in methods:
                acc = ev["accuracy"]
                c = ev["correct"]
                t = ev["total"]
                bar = "█" * int(acc * 20) + "░" * (20 - int(acc * 20))
                
                short_name = method.replace("Ours (Neuro-Symbolic)", "★ Ours")
                short_name = short_name.replace("multi_", "m_")
                
                lines.append(f"  │ {short_name:<25} {bar} {acc:>5.1%} ({c}/{t}) │")
            lines.append(f"  └{'─' * 58}┘")
        
        # ===== 逐题对比表 =====
        lines.append("\n  --- Per-Question Results ---\n")
        
        show_methods =["Ours (Neuro-Symbolic)", "multi_direct"]
        show_methods =[m for m in show_methods if m in all_evals]
        
        # 表头
        header = f"  {'Q':<5}"
        for m in show_methods:
            short = m.replace("Ours (Neuro-Symbolic)", "Ours")
            short = short.replace("multi_", "m.")
            header += f" {short:>8}"
        lines.append(header)
        lines.append("  " + "─" * (5 + 9 * len(show_methods)))
        
        for q in questions:
            row = f"  {q['question_id']:<5}"
            for m in show_methods:
                ev = all_evals[m]
                pq = next(
                    (p for p in ev["per_question"] 
                     if p["question_id"] == q["question_id"]), None
                )
                if pq and pq["correct"]:
                    row += f"    {'✓':>4}"
                elif pq:
                    row += f"    {'✗':>4}"
                else:
                    row += f"    {'?':>4}"
            lines.append(row)
        
        # ===== 关键分析 =====
        lines.append("\n  --- Key Findings ---\n")
        
        ours_key = "Ours (Neuro-Symbolic)"
        baseline_key = "multi_direct"
        
        if ours_key in all_evals and baseline_key in all_evals:
            ours_acc = all_evals[ours_key]["accuracy"]
            baseline_acc = all_evals[baseline_key]["accuracy"]
            delta = ours_acc - baseline_acc
            
            lines.append(f"  vs Baseline ({baseline_key}):")
            lines.append(f"    Ours: {ours_acc:.1%} vs Baseline: {baseline_acc:.1%} (Δ = {delta:+.1%})")
            
            if delta > 0:
                lines.append(f"    → ★ Our neuro-symbolic framework adds {delta:+.1%} over")
                lines.append(f"      the baseline WITH SAME VISUAL INPUT.")
                lines.append(f"      This improvement comes purely from structured reasoning.")
            elif delta == 0:
                lines.append(f"    → Tied with baseline.")
            else:
                lines.append(f"    → ⚠ Underperforms baseline. Investigate errors.")
            
            # 找到只有我们答对的题
            ours_correct = {p["question_id"] for p in all_evals[ours_key]["per_question"] if p["correct"]}
            baseline_correct = {p["question_id"] for p in all_evals[baseline_key]["per_question"] if p["correct"]}
            
            unique_ours = ours_correct - baseline_correct
            if unique_ours:
                lines.append(f"\n  ★ Questions ONLY our method answered correctly: {unique_ours}")
                for q_id in unique_ours:
                    q_data = next(q for q in questions if q["question_id"] == q_id)
                    lines.append(f"    {q_id}: {q_data['question'][:65]}...")
        
        # ===== 总结表 =====
        lines.append("\n  " + "=" * 50)
        lines.append("  SUMMARY TABLE (for paper)")
        lines.append("  " + "=" * 50)
        lines.append(f"\n  {'Method':<30} {'Accuracy':>10} {'Frames':>8}")
        lines.append("  " + "─" * 50)
        
        # 按准确率排序
        sorted_evals = sorted(all_evals.items(), 
                               key=lambda x: x[1]["accuracy"], reverse=True)
        for method, ev in sorted_evals:
            if "Ours" in method:
                method_display = f"★ {method}"
            else:
                method_display = f"  {method}"
            lines.append(f"  {method_display:<30} {ev['accuracy']:>9.1%} {'N':>8}")
        
        return "\n".join(lines)


# ================================================================
#  运行入口
# ================================================================

if __name__ == "__main__":
    
    # 路径配置
    video_path = "/Users/zehao/Desktop/HAI/MultiParty/Social-IQ-2/siq2/video/i3itBKdwE7M.mp4"
    qa_file_path = "/Users/zehao/Desktop/HAI/MultiParty/Social-IQ-2/siq2/qa/qa_train.json"
    pipeline_result_path = "/Users/zehao/Desktop/HAI/MultiParty/code/output_full_results.json"
    comparison_output_path = "/Users/zehao/Desktop/HAI/MultiParty/code/fair_comparison_results.json"
    
    os.makedirs(os.path.dirname(comparison_output_path), exist_ok=True)
    
    # 运行公平对比
    experiment = FairComparisonExperiment()
    results = experiment.run(
        video_path=video_path,
        qa_file_path=qa_file_path,
        pipeline_result_path=pipeline_result_path,
        output_path=comparison_output_path,
        num_sample_frames=50    # 采样8帧
        # api_delay=2.0           # API调用间隔
    )

  FAIR COMPARISON EXPERIMENT
  Pipeline and all baselines see the SAME frames

Loaded 29 questions for video: i3itBKdwE7M

--- Sampling 50 frames ---
Video: 640x360, 29FPS, 1799 frames, 62.0s
Sampled 50 frames at indices: [89, 122, 155, 188, 221, 254, 287, 320, 353, 386, 419, 452, 485, 518, 551, 584, 617, 651, 684, 717, 750, 783, 816, 849, 882, 915, 948, 981, 1014, 1047, 1080, 1113, 1146, 1180, 1213, 1246, 1279, 1312, 1345, 1378, 1411, 1444, 1477, 1510, 1543, 1576, 1609, 1642, 1675, 1709]
  All baselines will see frames: [89, 122, 155, 188, 221, 254, 287, 320, 353, 386, 419, 452, 485, 518, 551, 584, 617, 651, 684, 717, 750, 783, 816, 849, 882, 915, 948, 981, 1014, 1047, 1080, 1113, 1146, 1180, 1213, 1246, 1279, 1312, 1345, 1378, 1411, 1444, 1477, 1510, 1543, 1576, 1609, 1642, 1675, 1709]

--- Pipeline Results ---
  Loaded 5 pipeline answers

--- Running VLM Baseline ---
  Baseline sees 50 frames directly

  [i3itBKdwE7M_q15_6] Do the three girls at the end seem to be having a good ...


KeyboardInterrupt: 

# VLM直接推理整个数据集

In [23]:
# ================================================================
#  全集评估模块 + VLM Baseline 公平对比
#  
#  功能：
#  1. 从 pipeline 跑出的 JSON 结果中读取需要测试的视频列表
#  2. 按视频提取帧并交由 VLM (Baseline) 直接推理
#  3. 收集两边的所有答案，进行全数据集级别的评估
#  4. 最终输出全局的 Accuracy 对比和错题分析
# ================================================================

import os
import json
import time
import base64
import re
from io import BytesIO
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass
from PIL import Image
import numpy as np
import cv2

import openai

# ================= API 配置 =================
API_KEY = "sk-bbf9f24ff4194920a43e15749a2dad29"           # ← 填写你的API Key
API_BASE = "https://dashscope.aliyuncs.com/compatible-mode/v1"  # ← 填写你的API Base URL
MODEL_NAME = "qwen-vl-plus-2025-07-10"

client = openai.OpenAI(api_key=API_KEY, base_url=API_BASE)

class VideoSampler:
    """统一的视频帧采样器，确保两边看到完全相同的帧"""
    def __init__(self, video_path: str):
        self.video_path = video_path
        
        cap = cv2.VideoCapture(video_path)
        self.fps = int(cap.get(cv2.CAP_PROP_FPS))
        self.width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        self.height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        self.total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        self.duration = self.total_frames / self.fps if self.fps > 0 else 0
        cap.release()
    
    def sample_frames(self, num_frames: int = 8, strategy: str = "uniform") -> List[dict]:
        if strategy == "uniform":
            start = int(self.total_frames * 0.05)
            end = int(self.total_frames * 0.95)
            indices = np.linspace(start, end, num_frames, dtype=int).tolist()
        elif strategy == "keypoints":
            positions =[0.15, 0.25, 0.35, 0.45, 0.55, 0.65, 0.75, 0.85]
            indices =[int(self.total_frames * p) for p in positions[:num_frames]]
        else:
            raise ValueError(f"Unknown strategy: {strategy}")
        
        indices = sorted(set(indices))
        return self._extract_frames(indices)
    
    def _extract_frames(self, indices: List[int]) -> List[dict]:
        cap = cv2.VideoCapture(self.video_path)
        frames =[]
        for idx in indices:
            idx = max(0, min(idx, self.total_frames - 1))
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if ret:
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                pil = Image.fromarray(rgb)
                frames.append({
                    "frame_idx": idx,
                    "pil_image": pil,
                    "timestamp": idx / self.fps if self.fps > 0 else 0
                })
        cap.release()
        return frames

# ================================================================
#  多帧图像处理工具
# ================================================================

class MultiFrameProcessor:
    @staticmethod
    def encode_image(pil_image: Image.Image, max_size: int = 768) -> str:
        w, h = pil_image.size
        if max(w, h) > max_size:
            ratio = max_size / max(w, h)
            pil_image = pil_image.resize((int(w * ratio), int(h * ratio)), Image.LANCZOS)
        buf = BytesIO()
        pil_image.save(buf, format="JPEG", quality=85)
        return base64.b64encode(buf.getvalue()).decode("utf-8")

# ================================================================
#  VLM Baseline（多帧直接推理版本）
# ================================================================

class FairVLMBaseline:
    def __init__(self, model_name: str = MODEL_NAME, max_retries: int = 3):
        self.model_name = model_name
        self.max_retries = max_retries
        self.processor = MultiFrameProcessor()
    
    def _call_api(self, messages: list, temperature: float = 0.1, max_tokens: int = 2000) -> str:
        for attempt in range(self.max_retries):
            try:
                response = client.chat.completions.create(
                    model=self.model_name,
                    messages=messages,
                    temperature=temperature,
                    max_tokens=max_tokens
                )
                return response.choices[0].message.content
            except Exception as e:
                print(f"    API error (attempt {attempt+1}): {e}")
                if attempt < self.max_retries - 1:
                    time.sleep(2 ** attempt)
                else:
                    return f"[Error: {str(e)}]"
    
    def _parse_response(self, raw: str) -> dict:
        try:
            match = re.search(r'\{[\s\S]*?\}', raw)
            if match:
                result = json.loads(match.group())
                
                # ====== 强防御：确保提取出的是整型 ======
                val = result.get("selected_answer_index", -1)
                if isinstance(val, list):
                    val = val[0] if len(val) > 0 else -1
                try:
                    result["selected_answer_index"] = int(val)
                except (ValueError, TypeError):
                    result["selected_answer_index"] = -1
                # ========================================
                
                return result
        except json.JSONDecodeError:
            pass
        
        idx_match = re.search(r'(?:select|answer|index)[^\d]*(\d+)', raw, re.IGNORECASE)
        if idx_match:
            return {"selected_answer_index": int(idx_match.group(1)), "reasoning": raw}
        
        return {"selected_answer_index": -1, "reasoning": raw, "raw": raw}
    
    def _multi_image_input(self, frames: List[dict], system_prompt: str, user_prompt: str,
                            max_images: int = 8, temperature: float = 0.1) -> str:
        if len(frames) > max_images:
            indices = np.linspace(0, len(frames)-1, max_images, dtype=int)
            frames =[frames[i] for i in indices]
        
        image_contents =[]
        for f in frames:
            b64 = self.processor.encode_image(f["pil_image"], max_size=512)
            image_contents.append({
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{b64}"}
            })
        
        frame_desc = ", ".join([f"Frame {f['frame_idx']} ({f['timestamp']:.1f}s)" for f in frames])
        messages =[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content":[
                *image_contents,
                {"type": "text", "text": f"The above {len(frames)} images are sampled from a video at: {frame_desc}\n\n{user_prompt}"}
            ]}
        ]
        return self._call_api(messages, temperature, max_tokens=2000)
    
    def _build_prompts(self, question: str, choices: List[dict], num_frames: int) -> Tuple[str, str]:
        choices_text = "\n".join([f"  [{c['index']}] {c['text']}" for c in choices])
        frame_context = f"You are viewing {num_frames} frames sampled from a video of a multi-party social interaction. Consider the temporal progression across frames."
        system_prompt = f"""You are analyzing a social interaction scene.
{frame_context}
Answer the multiple-choice question by selecting the best answer.

Respond ONLY in this JSON format:
{{"selected_answer_index": <int>, "selected_answer_text": "<text>", "reasoning": "<brief reasoning>"}}"""
        user_prompt = f"""Question: {question}\n\nAnswer Choices:\n{choices_text}\n\nSelect the best answer by index."""
        return system_prompt, user_prompt
    
    def run_all_baselines(self, questions: List[dict], frames: List[dict], delay: float = 2.0) -> Dict[str, List[dict]]:
        baseline_configs = [("multi_direct", "multi_image", "direct")]
        results = {name:[] for name, _, _ in baseline_configs}
        
        for q_idx, q in enumerate(questions):
            print(f"    Q: {q['question'][:45]}...", end=" ", flush=True)
            for name, input_mode, reasoning_mode in baseline_configs:
                sys_prompt, user_prompt = self._build_prompts(q["question"], q["answers"], len(frames))
                raw_ans = self._multi_image_input(frames, sys_prompt, user_prompt)
                ans = self._parse_response(raw_ans)
                ans["method"] = name
                results[name].append(ans)
                
                # ====== 强防御：确保转换为整型 ======
                sel_idx = ans.get("selected_answer_index", -1)
                if isinstance(sel_idx, list):
                    sel_idx = sel_idx[0] if len(sel_idx) > 0 else -1
                try:
                    sel_idx = int(sel_idx)
                except (ValueError, TypeError):
                    sel_idx = -1
                # ==================================
                
                if 0 <= sel_idx < len(q["answers"]):
                    is_correct = q["answers"][sel_idx].get("is_correct", False)
                    mark = "✓" if is_correct else "✗"
                else:
                    mark = "?"
                print(f"[{mark}]", end=" ")
                time.sleep(delay)
            print() # 换行
        return results

# ================================================================
#  评估器
# ================================================================

class SocialIQEvaluator:
    @staticmethod
    def parse_qa_file(filepath: str) -> List[dict]:
        """解析 Social-IQ 2.0 的 JSON/JSONL QA文件"""
        questions =[]
        with open(filepath, 'r', encoding='utf-8') as f:
            content = f.read().strip()
            if content.startswith('['):
                data_list = json.loads(content)
            else:
                data_list =[json.loads(line) for line in content.split('\n') if line.strip()]
        
        for data in data_list:
            q_id = data.get("qid")
            q_text = data.get("q")
            correct_idx = data.get("answer_idx", -1)
            current_q = {
                "question_id": q_id,
                "question": q_text,
                "vid_name": data.get("vid_name"),
                "answers":[]
            }
            for i in range(4):
                ans_key = f"a{i}"
                if ans_key in data:
                    is_correct = (i == correct_idx)
                    current_q["answers"].append({
                        "index": i, 
                        "label": 'a' if is_correct else 'i',
                        "text": data[ans_key], 
                        "is_correct": is_correct
                    })
            questions.append(current_q)
        return questions
    
    @staticmethod
    def evaluate_batch(questions: List[dict], model_answers: List[dict]) -> dict:
        results =[]
        correct = 0
        for q, ans in zip(questions, model_answers):
            # ====== 强防御：确保转换为整型 ======
            sel_idx = ans.get("selected_answer_index", -1)
            if isinstance(sel_idx, list):
                sel_idx = sel_idx[0] if len(sel_idx) > 0 else -1
            try:
                sel_idx = int(sel_idx)
            except (ValueError, TypeError):
                sel_idx = -1
            # ==================================
            
            if 0 <= sel_idx < len(q["answers"]):
                is_correct = q["answers"][sel_idx]["is_correct"]
            else:
                is_correct = False
            
            results.append({
                "question_id": q["question_id"],
                "correct": is_correct,
            })
            if is_correct: correct += 1
        
        total = len(questions)
        return {
            "accuracy": correct / total if total > 0 else 0,
            "correct": correct,
            "total": total,
            "per_question": results
        }

# ================================================================
#  全集批量对比实验
# ================================================================

class DatasetFairComparison:
    def __init__(self):
        self.evaluator = SocialIQEvaluator()
        self.vlm = FairVLMBaseline(model_name=MODEL_NAME)
    
    def run_dataset(self, video_dir: str, qa_file_path: str, pipeline_result_path: str,
                    output_path: str, num_sample_frames: int = 8, api_delay: float = 2.0):
        
        print("=" * 70)
        print("  FAIR COMPARISON EXPERIMENT (FULL DATASET)")
        print("=" * 70)

        # 1. 加载并整理所有的问题
        print("Parsing QA Database...")
        all_questions = self.evaluator.parse_qa_file(qa_file_path)
        qs_by_vid = {}
        for q in all_questions:
            qs_by_vid.setdefault(q["vid_name"],[]).append(q)
        
        # 2. 加载 pipeline 结果
        if not os.path.exists(pipeline_result_path):
            print(f"Error: Pipeline result JSON not found at {pipeline_result_path}")
            return
        
        with open(pipeline_result_path, 'r') as f:
            pipeline_data = json.load(f)
        
        pipeline_videos = pipeline_data.get("results",[])
        total_videos = len(pipeline_videos)
        print(f"Found {total_videos} videos in pipeline results.")

        # 全局统计容器
        global_questions =[]
        global_pipeline_answers = []
        global_baseline_answers = {"multi_direct":[]}

        # 3. 遍历每个被 Pipeline 处理过的视频
        for idx, item in enumerate(pipeline_videos):
            vid_name = item.get("vid_name")
            
            # 如果 Pipeline 处理出错了或者没有做题，跳过
            if "error" in item or not item.get("qa_results"):
                print(f"[{idx+1}/{total_videos}] Skipped {vid_name} (Pipeline had error or 0 questions)")
                continue
            
            # 找到对应的视频文件
            video_path = None
            for ext in['.mp4', '.avi', '.mov', '.mkv']:
                p = os.path.join(video_dir, f"{vid_name}{ext}")
                if os.path.exists(p):
                    video_path = p
                    break
            
            if not video_path:
                print(f"[{idx+1}/{total_videos}] Skipped {vid_name} (Video file not found locally)")
                continue
            
            # 获取这个视频对应的问题
            vid_questions = qs_by_vid.get(vid_name,[])
            if not vid_questions:
                print(f"[{idx+1}/{total_videos}] Skipped {vid_name} (No QA pairs found in DB)")
                continue
            
            print(f"\n[{idx+1}/{total_videos}] 🎥 Video: {vid_name} ({len(vid_questions)} questions)")

            # A. 提取视频帧
            sampler = VideoSampler(video_path)
            frames = sampler.sample_frames(num_frames=num_sample_frames, strategy="uniform")
            
            # B. 运行 VLM Baseline
            baseline_res = self.vlm.run_all_baselines(vid_questions, frames, delay=api_delay)
            
            # C. 提取 Pipeline 对应的答案
            pipe_qa_dict = {qa["question_id"]: qa["predicted_index"] for qa in item["qa_results"]}
            pipe_answers =[]
            for q in vid_questions:
                pred_idx = pipe_qa_dict.get(q["question_id"], -1)
                pipe_answers.append({"selected_answer_index": pred_idx})
            
            # 存入全局列表
            global_questions.extend(vid_questions)
            global_pipeline_answers.extend(pipe_answers)
            global_baseline_answers["multi_direct"].extend(baseline_res["multi_direct"])

        # 4. 全局评估
        print("\n" + "=" * 70)
        print("  FULL DATASET EVALUATION")
        print("=" * 70)
        
        all_evals = {}
        all_evals["Ours (Neuro-Symbolic)"] = self.evaluator.evaluate_batch(global_questions, global_pipeline_answers)
        all_evals["multi_direct"] = self.evaluator.evaluate_batch(global_questions, global_baseline_answers["multi_direct"])
        
        report = self._generate_report(all_evals, global_questions, num_sample_frames)
        print(report)
        
        # 5. 保存结果
        output = {
            "experiment_config": {
                "num_videos_processed": len(pipeline_videos),
                "num_questions_evaluated": len(global_questions),
                "num_frames_sampled_per_video": num_sample_frames,
                "baseline_model": MODEL_NAME
            },
            "evaluations": {
                method: {"accuracy": ev["accuracy"], "correct": ev["correct"], "total": ev["total"]}
                for method, ev in all_evals.items()
            },
            "report": report
        }
        
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(output, f, indent=2, ensure_ascii=False, default=str)
        print(f"\n✅ Full dataset results saved to: {output_path}")

    def _generate_report(self, all_evals: dict, questions: list, num_frames: int) -> str:
        lines =[]
        lines.append(f"Tested on {len(questions)} Total Questions")
        lines.append("-" * 50)
        
        groups = {"Our Method":[], "VLM Baseline (Direct Multi-Image)":[]}
        for method, ev in all_evals.items():
            if "Ours" in method: groups["Our Method"].append((method, ev))
            elif "multi" in method: groups["VLM Baseline (Direct Multi-Image)"].append((method, ev))
        
        for group_name, methods in groups.items():
            if not methods: continue
            lines.append(f"\n[ {group_name} ]")
            for method, ev in methods:
                acc = ev["accuracy"]
                c, t = ev["correct"], ev["total"]
                bar = "█" * int(acc * 20) + "░" * (20 - int(acc * 20))
                short_name = method.replace("Ours (Neuro-Symbolic)", "★ Pipeline").replace("multi_direct", "Baseline")
                lines.append(f"  {short_name:<12} {bar} {acc:>5.1%} ({c}/{t})")
        
        lines.append("\n" + "-" * 50)
        ours_key, baseline_key = "Ours (Neuro-Symbolic)", "multi_direct"
        if ours_key in all_evals and baseline_key in all_evals:
            o_acc, b_acc = all_evals[ours_key]["accuracy"], all_evals[baseline_key]["accuracy"]
            delta = o_acc - b_acc
            lines.append(f"Final Delta: Ours vs Baseline = {delta:+.1%} Accuracy")
            
            # 统计两者答题交叉情况
            ours_corr = {p["question_id"] for p in all_evals[ours_key]["per_question"] if p["correct"]}
            base_corr = {p["question_id"] for p in all_evals[baseline_key]["per_question"] if p["correct"]}
            
            both_correct = ours_corr & base_corr
            ours_only = ours_corr - base_corr
            base_only = base_corr - ours_corr
            
            lines.append(f"  - Both Correct:     {len(both_correct)}")
            lines.append(f"  - ★ Ours Only:     {len(ours_only)}")
            lines.append(f"  - Baseline Only:    {len(base_only)}")
            
        return "\n".join(lines)


# ================================================================
#  运行入口
# ================================================================

if __name__ == "__main__":
    
    # 【重要】将这里的 video_path 替换为你的 视频所在文件夹 路径！
    VIDEO_DIR = "/Users/zehao/Desktop/HAI/MultiParty/Social-IQ-2/siq2/video" 
    
    QA_FILE_PATH = "/Users/zehao/Desktop/HAI/MultiParty/Social-IQ-2/siq2/qa/all.json"
    PIPELINE_RESULT_PATH = "./results/m4_prompt_results.json" 
    COMPARISON_OUTPUT_PATH = "./comparison_200.json"
    
    os.makedirs(os.path.dirname(COMPARISON_OUTPUT_PATH), exist_ok=True)
    
    experiment = DatasetFairComparison()
    experiment.run_dataset(
        video_dir=VIDEO_DIR,
        qa_file_path=QA_FILE_PATH,
        pipeline_result_path=PIPELINE_RESULT_PATH,
        output_path=COMPARISON_OUTPUT_PATH,
        num_sample_frames=20,   # 给 VLM 每条视频抽取的帧数
        api_delay=2.0           # 防止触发大模型限流
    )

  FAIR COMPARISON EXPERIMENT (FULL DATASET)
Parsing QA Database...
Found 29 videos in pipeline results.

[1/29] 🎥 Video: ZP8ACbJ677I (5 questions)
    Q: As a whole, what are the man's emotions?... [✗] 
    Q: How does the man feel about the mascara?... [✗] 
    Q: How does the man feel about the woman?... [✗] 
    Q: How does the woman feel about the mascara?... [✓] 
    Q: To whom is the man talking?... [✗] 

[2/29] 🎥 Video: SWNXZbasPvQ (11 questions)
    Q: Does the man in plaid agree with the man in b... [✗] 
    Q: Does the man on the far left agree with the i... [✗] 
    Q: How does the man feel about what he is discus... [✓] 
    Q: How does the man in black make everyone feel?... [✓] 
    Q: How does the man on the far right make everyo... [✓] 
    Q: How does the woman feel about what the man do... [✗] 
    Q: The lady's laugh suggests that she is... [✓] 
    Q: The smiles and laughter of all the people in ... [✗] 
    Q: Why does the man believe power poses work?... [✓] 
    